#  SINGLE GPU LORA AND QLORA FINE-TUNING 

In [ ]:
# CELL-1: BASIC SETUP
import os
import sys

# --- 1. INSTALL STANDARD STACK ---
!pip install -U transformers peft bitsandbytes accelerate datasets pynvml trl

import torch
import gc
import time
import threading
import csv
import pynvml
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    BitsAndBytesConfig, 
    TrainingArguments
)
from peft import (
    LoraConfig, 
    get_peft_model, 
    prepare_model_for_kbit_training, 
    TaskType
)
from trl import SFTTrainer
from datasets import load_dataset, Dataset
from huggingface_hub import login

# --- 2. LOGIN & CONFIG ---
# (Kaggle Secrets or Manual)
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    token = user_secrets.get_secret("HF_TOKEN")
    login(token)
except:
    login(os.getenv("HF_TOKEN"))

# --- HARDWARE & STRATEGY ---
HARDWARE_ENV = "T4"  # Change to "H100" on server

ROW_COUNTS = {
    "1B":  {"rag": 15000, "summ": 5000, "chat": 10000},
    "3B":  {"rag": 10000, "summ": 5000, "chat": 10000},
    "8B":  {"rag": 5000,  "summ": 5000, "chat": 10000}, 
    "30B": {"rag": 2500,  "summ": 5000, "chat": 10000}, 
}

if HARDWARE_ENV == "T4_DUAL"
    EXPERIMENTS = [
        ("Qwen/Qwen2.5-1.5B-Instruct", "1.5B", True, True),
        ("Qwen/Qwen2.5-3B-Instruct","3B", True, True),
        ("Qwen/Qwen2.5-7B-Instruct", "7B", True, False),  
        ("meta-llama/Llama-3.2-1B-Instruct", "1B", True, True),
        ("meta-llama/Llama-3.2-3B-Instruct", "3B", True, True),
        ("meta-llama/Llama-3.1-8B-Instruct", "7B", True, False),
    ]

In [ ]:
# ==================== CELL 2 - FIXED FORMATTING ====================
SYSTEM_PROMPTS = {
    "rag": """You are a factual assistant. Use ONLY the provided context to answer the question.
If the answer is not contained in the context, respond with "I do not have enough information." """,
    "summ": """Write exactly one sentence that serves as a news-style lead summarizing the most important event in the article.
Do not use multiple sentences. Do not add information not present in the article.""",
    "chat": """You are a helpful and coherent AI assistant."""
}

def format_data_using_chat_template(examples, tokenizer, task):
    """
    FIXED: This function now properly handles batched examples from datasets.map()
    examples is a dict of lists, not a Dataset object
    """
    texts = []
    system_msg = SYSTEM_PROMPTS[task]
    
    # ------------------ RAG (SQuAD) ------------------
    if task == "rag":
        # Access dict keys directly (examples is a dict, not a Dataset)
        contexts = examples["context"]
        questions = examples["question"]
        answers = examples["answers"]
        
        for ctx, q, ans in zip(contexts, questions, answers):
            try:
                if not ctx or not q or not ans:
                    continue
                
                # SQuAD answer extraction
                if isinstance(ans, dict) and "text" in ans:
                    answer_text = ans["text"][0] if ans["text"] else None
                else:
                    answer_text = str(ans)
                
                if not answer_text:
                    continue
                
                user_content = f"### Context:\n{ctx}\n\n### Question:\n{q}"
                conversation = [
                    {"role": "system", "content": system_msg},
                    {"role": "user", "content": user_content},
                    {"role": "assistant", "content": answer_text}
                ]
                texts.append(tokenizer.apply_chat_template(
                    conversation, tokenize=False, add_generation_prompt=False
                ))
            except Exception:
                continue
    
    # ------------------ SUMMARIZATION (XSum) ------------------
    elif task == "summ":
        # Access dict keys directly
        docs = examples["document"]
        summs = examples["summary"]
        
        for ctx, output in zip(docs, summs):
            if not ctx or not output: 
                continue
            
            user_content = f"### Article:\n{ctx}"
            conversation = [
                {"role": "system", "content": system_msg},
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": output}
            ]
            texts.append(tokenizer.apply_chat_template(
                conversation, tokenize=False, add_generation_prompt=False
            ))
    
    # ------------------ CHAT (UltraChat) ------------------
    elif task == "chat":
        # Access dict keys directly
        all_messages = examples["messages"]
        
        for conversation in all_messages:
            if not conversation: 
                continue
            
            conv_obj = []
            for msg in conversation:
                if isinstance(msg, dict):
                    conv_obj.append(msg.copy())
            
            if not conv_obj: 
                continue
            
            if conv_obj[0]["role"] != "system":
                conv_obj.insert(0, {"role": "system", "content": system_msg})
            
            texts.append(tokenizer.apply_chat_template(
                conv_obj, tokenize=False, add_generation_prompt=False
            ))
    
    return {"text": texts}  # Return as dict for datasets.map()



In [ ]:
# ==================== CELL 3 - FIXED DATA LOADING ====================
def load_and_prep_data(task, row_count, tokenizer):
    print(f"   ...Loading {task.upper()} ({row_count} rows)...")
    
    CONFIGS = {
        "rag":  ("squad", "train", "main"),
        "summ": ("EdinburghNLP/xsum", "train", "main"), 
        "chat": ("HuggingFaceH4/ultrachat_200k", "train_sft", "main") 
    }
    
    ds_id, default_split, rev = CONFIGS[task]
    
    try:
        # RAG: Direct load
        if task == "rag":
            print(f"      📥 Loading SQuAD dataset...")
            ds = load_dataset(ds_id, split=f"train[:{int(row_count * 1.0)}]", revision=rev)
            print(f"      ✅ Loaded {len(ds)} rows in seconds!")
        
        # Other tasks: Streaming
        else:
            try:
                ds_stream = load_dataset(ds_id, split=default_split, streaming=True, revision=rev)
            except ValueError:
                ds_stream = load_dataset(ds_id, split="train", streaming=True, revision=rev)
            
            safe_count = int(row_count * 1.0)
            data_head = []
            
            print(f"      📥 Streaming {safe_count} samples...")
            iter_count = 0
            max_iterations = safe_count * 2
            
            for item in ds_stream:
                data_head.append(item)
                iter_count += 1
                
                if iter_count % 1000 == 0:
                    print(f"      ... {iter_count} collected")
                
                if len(data_head) >= safe_count:
                    break
                
                if iter_count > max_iterations:
                    print(f"      ⚠️ Safety limit at {iter_count}")
                    break
            
            if len(data_head) == 0:
                print(f"      ❌ No data collected")
                return None
                
            ds = Dataset.from_list(data_head)
            print(f"      ✅ Loaded {len(ds)} rows.")
            
    except Exception as e:
        print(f"      ❌ LOAD ERROR: {e}")
        import traceback
        traceback.print_exc()
        return None
    
    # FIXED: Return only the dataset, we'll format it differently
    return ds



In [ ]:
# ==================== CELL 4 - COMPLETELY FIXED TRAINING LOOP ====================
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

OUTPUT_DIR = "./fine_tuned_adapters"
LOG_FILE = "training_energy_log.csv"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- ENERGY MONITOR (Keep as is) ---
class EnergyMonitor:
    def __init__(self, interval=0.5):
        self.interval = interval
        self.running = False
        self.joules = 0.0
        self.peak_watts = 0.0
        self.duration = 0.0
        try:
            pynvml.nvmlInit()
            self.handle = pynvml.nvmlDeviceGetHandleByIndex(0)
            self.available = True
        except:
            self.available = False
            print("⚠️ pynvml failed (no GPU?). Energy tracking disabled.")
    
    def _monitor(self):
        start_time = time.time()
        while self.running and self.available:
            try:
                power = pynvml.nvmlDeviceGetPowerUsage(self.handle) / 1000.0
                if power > self.peak_watts: self.peak_watts = power
                self.joules += power * self.interval
            except: pass
            time.sleep(self.interval)
        self.duration = time.time() - start_time

    def start(self):
        if self.available:
            self.running = True
            self.thread = threading.Thread(target=self._monitor)
            self.thread.start()

    def stop(self):
        self.running = False
        if self.available: self.thread.join()
        avg_watts = (self.joules / self.duration) if self.duration > 0 else 0
        return self.joules, avg_watts, self.duration

# --- CSV INIT ---
if not os.path.exists(LOG_FILE):
    with open(LOG_FILE, "w") as f:
        csv.writer(f).writerow(["Model", "Size", "Method", "Task", "Joules", "Watts", "Seconds", "Rows"])

# --- MASTER LOOP ---
for model_id, size_label, do_lora, do_qlora in EXPERIMENTS:
    
    modes = []
    if do_qlora: modes.append("QLoRA_INT4")
    if do_lora: modes.append("LoRA_FP16")
    
    for task in ["rag", "summ", "chat"]:
        n_rows = ROW_COUNTS[size_label][task]
        
        for tune_type in modes:
            print(f"\n{'='*60}\n🤖 {size_label} | {tune_type} | TASK: {task}\n{'='*60}")
            
            is_qlora = (tune_type == "QLoRA_INT4")
            
            if is_qlora:
                bnb_config = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_quant_type="nf4",
                    bnb_4bit_compute_dtype=torch.float16,
                    bnb_4bit_use_double_quant=True,
                )
            else:
                bnb_config = None
            
            try:
                gc.collect()
                torch.cuda.empty_cache()
                
                # --- LOAD TOKENIZER ---
                tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
                tokenizer.pad_token = tokenizer.eos_token
                tokenizer.padding_side = "right"
                
                # --- LOAD MODEL ---
                model = AutoModelForCausalLM.from_pretrained(
                    model_id,
                    quantization_config=bnb_config,
                    torch_dtype=torch.float16,
                    device_map="auto",
                    trust_remote_code=True
                )
                
                if is_qlora:
                    model = prepare_model_for_kbit_training(model)
                
                # --- ATTACH LoRA ---
                peft_config = LoraConfig(
                    r=16,
                    lora_alpha=16,
                    lora_dropout=0.05,
                    bias="none",
                    task_type="CAUSAL_LM",
                    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", 
                                   "gate_proj", "up_proj", "down_proj"]
                )
                model = get_peft_model(model, peft_config)
                model.print_trainable_parameters()
                
                # --- LOAD DATA ---
                dataset = load_and_prep_data(task, n_rows, tokenizer)
                if dataset is None or len(dataset) == 0: 
                    print(f"      ⚠️ Skipping due to data loading failure")
                    continue
                
                # --- FORMAT DATA ---
                print(f"      🔄 Formatting {len(dataset)} examples...")
                
                # FIXED: Use map() properly - it passes batches as dicts
                formatted_dataset = dataset.map(
                    lambda examples: format_data_using_chat_template(examples, tokenizer, task),
                    batched=True,
                    remove_columns=dataset.column_names,
                    desc="Formatting"
                )
                
                # Filter out empty examples
                formatted_dataset = formatted_dataset.filter(
                    lambda x: x["text"] is not None and len(x["text"].strip()) > 0
                )
                
                print(f"      ✅ {len(formatted_dataset)} valid examples after formatting")
                
                if len(formatted_dataset) == 0:
                    print(f"      ⚠️ No valid examples, skipping")
                    continue
                
                # --- TRAIN ---
                monitor = EnergyMonitor()
                monitor.start()
                
                # FIXED: Simplified SFTTrainer call - no deprecated params
                trainer = SFTTrainer(
                    model=model,
                    train_dataset=formatted_dataset,
                    args=TrainingArguments(
                        per_device_train_batch_size=4,
                        gradient_accumulation_steps=2
                        max_steps=-1,
                        num_train_epochs=1 ,
                        learning_rate=2e-4,
                        fp16=False, #true for lora training
                        bf16=True, #false for lora training
                        logging_steps=20,
                        output_dir="temp_trainer",
                        optim="paged_adamw_32bit",
                        gradient_checkpointing=True,
                        report_to="none",
                        save_strategy="no",
                    ),
                )
                
                print(f"      🚀 Starting training...")
                trainer.train()
                
                joules, watts, duration = monitor.stop()
                print(f"      ⚡ Energy: {joules:.2f} J | {watts:.2f} W | {duration:.2f} s")
                
                # Log
                with open(LOG_FILE, "a") as f:
                    csv.writer(f).writerow([
                        model_id, size_label, tune_type, task, 
                        f"{joules:.2f}", f"{watts:.2f}", f"{duration:.2f}", len(formatted_dataset)
                    ])
                
                # --- SAVE ---
                adapter_name = f"{size_label}_{tune_type}_{task}"
                save_path = os.path.join(OUTPUT_DIR, adapter_name)
                trainer.model.save_pretrained(save_path)
                tokenizer.save_pretrained(save_path)
                print(f"      ✅ Saved: {save_path}")
                
                # Cleanup
                del model, tokenizer, trainer, monitor, formatted_dataset
                gc.collect()
                torch.cuda.empty_cache()
                
            except Exception as e:
                print(f"      ❌ FAILED: {e}")
                import traceback
                traceback.print_exc()
                try: 
                    monitor.stop()
                except: 
                    pass
                gc.collect()
                torch.cuda.empty_cache()

print("\n" + "="*60)
print("✅ TRAINING COMPLETE!")
print(f"📊 Results logged to: {LOG_FILE}")
print(f"💾 Adapters saved to: {OUTPUT_DIR}")

#  DUAL GPU LORA AND QLORA FINETUNING

In [ ]:
# CELL-1 BASIC SETUP
import os
import sys

# ⭐ DUAL GPU SETUP - REMOVE SINGLE GPU RESTRICTION
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # ❌ REMOVE THIS LINE
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# --- 1. INSTALL STANDARD STACK ---
!pip install -U transformers peft bitsandbytes accelerate datasets pynvml trl

import torch
import gc
import time
import threading
import csv
import pynvml
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    BitsAndBytesConfig, 
    TrainingArguments
)
from peft import (
    LoraConfig, 
    get_peft_model, 
    prepare_model_for_kbit_training, 
    TaskType
)
from trl import SFTTrainer
from datasets import load_dataset, Dataset
from huggingface_hub import login

# ⭐ VERIFY DUAL GPU SETUP
print(f"🔍 GPUs Available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"   GPU {i}: {torch.cuda.get_device_name(i)}")

# --- LOGIN ---
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    print("✅ Using Kaggle Secrets Token")
except:
    hf_token = "HF_TOKEN_PLACEHOLDER" 
    if not hf_token:
        hf_token = os.getenv("HF_TOKEN")

if hf_token:
    login(token=hf_token, add_to_git_credential=True)
else:
    print("❌ No token found! Please set HF_TOKEN.")

# --- HARDWARE & STRATEGY ---
HARDWARE_ENV = "T4_DUAL"  # Changed to indicate dual GPU

ROW_COUNTS = {
    "1B":  {"rag": 15000, "summ": 5000, "chat": 10000},
    "3B":  {"rag": 10000, "summ": 5000, "chat": 10000},
    "7B":  {"rag": 5000,  "summ": 5000, "chat": 10000}, 
    "14B": {"rag": 2500,  "summ": 5000, "chat": 10000}, 
}

if HARDWARE_ENV == "T4_DUAL":
    EXPERIMENTS = [
        ("Qwen/Qwen2.5-1.5B-Instruct", "1.5B", False, False),
        ("Qwen/Qwen2.5-3B-Instruct","3B", False, False),
        ("Qwen/Qwen2.5-7B-Instruct", "7B", False, True),  
        ("meta-llama/Llama-3.2-1B-Instruct", "1B", False, False),
        ("meta-llama/Llama-3.2-3B-Instruct", "3B", False, False),
        ("meta-llama/Llama-3.1-8B-Instruct", "7B", False, True),
    ]

In [ ]:
# ==================== CELL 2 - FIXED FORMATTING ====================
SYSTEM_PROMPTS = {
    "rag": """You are a factual assistant. Use ONLY the provided context to answer the question.
If the answer is not contained in the context, respond with "I do not have enough information." """,
    "summ": """Write exactly one sentence that serves as a news-style lead summarizing the most important event in the article.
Do not use multiple sentences. Do not add information not present in the article.""",
    "chat": """You are a helpful and coherent AI assistant."""
}

def format_data_using_chat_template(examples, tokenizer, task):
    """
    FIXED: This function now properly handles batched examples from datasets.map()
    examples is a dict of lists, not a Dataset object
    """
    texts = []
    system_msg = SYSTEM_PROMPTS[task]
    
    # ------------------ RAG (SQuAD) ------------------
    if task == "rag":
        # Access dict keys directly (examples is a dict, not a Dataset)
        contexts = examples["context"]
        questions = examples["question"]
        answers = examples["answers"]
        
        for ctx, q, ans in zip(contexts, questions, answers):
            try:
                if not ctx or not q or not ans:
                    continue
                
                # SQuAD answer extraction
                if isinstance(ans, dict) and "text" in ans:
                    answer_text = ans["text"][0] if ans["text"] else None
                else:
                    answer_text = str(ans)
                
                if not answer_text:
                    continue
                
                user_content = f"### Context:\n{ctx}\n\n### Question:\n{q}"
                conversation = [
                    {"role": "system", "content": system_msg},
                    {"role": "user", "content": user_content},
                    {"role": "assistant", "content": answer_text}
                ]
                texts.append(tokenizer.apply_chat_template(
                    conversation, tokenize=False, add_generation_prompt=False
                ))
            except Exception:
                continue
    
    # ------------------ SUMMARIZATION (XSum) ------------------
    elif task == "summ":
        # Access dict keys directly
        docs = examples["document"]
        summs = examples["summary"]
        
        for ctx, output in zip(docs, summs):
            if not ctx or not output: 
                continue
            
            user_content = f"### Article:\n{ctx}"
            conversation = [
                {"role": "system", "content": system_msg},
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": output}
            ]
            texts.append(tokenizer.apply_chat_template(
                conversation, tokenize=False, add_generation_prompt=False
            ))
    
    # ------------------ CHAT (UltraChat) ------------------
    elif task == "chat":
        # Access dict keys directly
        all_messages = examples["messages"]
        
        for conversation in all_messages:
            if not conversation: 
                continue
            
            conv_obj = []
            for msg in conversation:
                if isinstance(msg, dict):
                    conv_obj.append(msg.copy())
            
            if not conv_obj: 
                continue
            
            if conv_obj[0]["role"] != "system":
                conv_obj.insert(0, {"role": "system", "content": system_msg})
            
            texts.append(tokenizer.apply_chat_template(
                conv_obj, tokenize=False, add_generation_prompt=False
            ))
    
    return {"text": texts}  # Return as dict for datasets.map()



In [ ]:
# ==================== CELL 3 - FIXED DATA LOADING ====================
def load_and_prep_data(task, row_count, tokenizer):
    print(f"   ...Loading {task.upper()} ({row_count} rows)...")
    
    CONFIGS = {
        "rag":  ("squad", "train", "main"),
        "summ": ("EdinburghNLP/xsum", "train", "main"), 
        "chat": ("HuggingFaceH4/ultrachat_200k", "train_sft", "main") 
    }
    
    ds_id, default_split, rev = CONFIGS[task]
    
    try:
        # RAG: Direct load
        if task == "rag":
            print(f"      📥 Loading SQuAD dataset...")
            ds = load_dataset(ds_id, split=f"train[:{int(row_count * 1.0)}]", revision=rev)
            print(f"      ✅ Loaded {len(ds)} rows in seconds!")
        
        # Other tasks: Streaming
        else:
            try:
                ds_stream = load_dataset(ds_id, split=default_split, streaming=True, revision=rev)
            except ValueError:
                ds_stream = load_dataset(ds_id, split="train", streaming=True, revision=rev)
            
            safe_count = int(row_count * 1.0)
            data_head = []
            
            print(f"      📥 Streaming {safe_count} samples...")
            iter_count = 0
            max_iterations = safe_count * 2
            
            for item in ds_stream:
                data_head.append(item)
                iter_count += 1
                
                if iter_count % 1000 == 0:
                    print(f"      ... {iter_count} collected")
                
                if len(data_head) >= safe_count:
                    break
                
                if iter_count > max_iterations:
                    print(f"      ⚠️ Safety limit at {iter_count}")
                    break
            
            if len(data_head) == 0:
                print(f"      ❌ No data collected")
                return None
                
            ds = Dataset.from_list(data_head)
            print(f"      ✅ Loaded {len(ds)} rows.")
            
    except Exception as e:
        print(f"      ❌ LOAD ERROR: {e}")
        import traceback
        traceback.print_exc()
        return None
    
    # FIXED: Return only the dataset, we'll format it differently
    return ds



In [ ]:
# ==================== CELL 4 - DUAL GPU TRAINING LOOP ====================
OUTPUT_DIR = "./fine_tuned_adapters"
LOG_FILE = "training_energy_log.csv"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- DUAL GPU ENERGY MONITOR ---
class DualGPUEnergyMonitor:
    def __init__(self, interval=0.5):
        self.interval = interval
        self.running = False
        self.joules = 0.0
        self.peak_watts = 0.0
        self.duration = 0.0
        self.gpu_handles = []
        
        try:
            pynvml.nvmlInit()
            num_gpus = pynvml.nvmlDeviceGetCount()
            for i in range(num_gpus):
                self.gpu_handles.append(pynvml.nvmlDeviceGetHandleByIndex(i))
            self.available = True
            print(f"⚡ Energy monitoring enabled for {num_gpus} GPUs")
        except:
            self.available = False
            print("⚠️ pynvml failed. Energy tracking disabled.")
    
    def _monitor(self):
        start_time = time.time()
        while self.running and self.available:
            try:
                total_power = 0.0
                for handle in self.gpu_handles:
                    power = pynvml.nvmlDeviceGetPowerUsage(handle) / 1000.0  # Watts
                    total_power += power
                
                if total_power > self.peak_watts:
                    self.peak_watts = total_power
                
                self.joules += total_power * self.interval
            except:
                pass
            time.sleep(self.interval)
        self.duration = time.time() - start_time

    def start(self):
        if self.available:
            self.running = True
            self.thread = threading.Thread(target=self._monitor)
            self.thread.start()

    def stop(self):
        self.running = False
        if self.available:
            self.thread.join()
        avg_watts = (self.joules / self.duration) if self.duration > 0 else 0
        return self.joules, avg_watts, self.duration

# --- CSV INIT ---
if not os.path.exists(LOG_FILE):
    with open(LOG_FILE, "w") as f:
        csv.writer(f).writerow(["Model", "Size", "Method", "Task", "Joules", "Watts", "Seconds", "Rows", "GPUs"])

# --- MASTER LOOP ---
for model_id, size_label, do_lora, do_qlora in EXPERIMENTS:
    
    modes = []
    #if do_lora: modes.append("LoRA_FP16")
    if do_qlora: modes.append("QLoRA_INT4")
    if do_lora: modes.append("LoRA_FP16")
    
    for task in ["rag", "summ", "chat"]:
        n_rows = ROW_COUNTS[size_label][task]
        
        for tune_type in modes:
            print(f"\n{'='*60}\n🤖 {size_label} | {tune_type} | TASK: {task}\n{'='*60}")
            
            is_qlora = (tune_type == "QLoRA_INT4")
            
            if is_qlora:
                bnb_config = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_quant_type="nf4",
                    bnb_4bit_compute_dtype=torch.float16,
                    bnb_4bit_use_double_quant=True,
                )
            else:
                bnb_config = None
            
            try:
                gc.collect()
                torch.cuda.empty_cache()
                
                # --- LOAD TOKENIZER ---
                tokenizer = AutoTokenizer.from_pretrained(
                    model_id, 
                    token=hf_token, 
                    trust_remote_code=True
                )
                tokenizer.pad_token = tokenizer.eos_token
                tokenizer.padding_side = "right"
                
                # ⭐ DUAL GPU CONFIGURATION
                # Calculate max memory per GPU (leave some headroom)
                max_memory = {
                    0: "13GB",  # GPU 0
                    1: "13GB",  # GPU 1
                }
                
                # --- LOAD MODEL WITH DUAL GPU SUPPORT ---
                model = AutoModelForCausalLM.from_pretrained(
                    model_id,
                    quantization_config=bnb_config,
                    torch_dtype=torch.float16,
                    device_map="auto",  # ⭐ Auto-distribute across GPUs
                    max_memory=max_memory,  # ⭐ Balanced memory
                    token=hf_token,
                    trust_remote_code=True,
                    offload_folder="offload",  # Optional: CPU offload if needed
                )
                
                print(f"   📊 Model device map: {model.hf_device_map}")
                
                if is_qlora:
                    model = prepare_model_for_kbit_training(model)
                
                # --- ATTACH LoRA ---
                peft_config = LoraConfig(
                    r=16,
                    lora_alpha=16,
                    lora_dropout=0.05,
                    bias="none",
                    task_type="CAUSAL_LM",
                    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", 
                                   "gate_proj", "up_proj", "down_proj"]
                )
                model = get_peft_model(model, peft_config)
                model.print_trainable_parameters()
                
                # --- LOAD DATA ---
                dataset = load_and_prep_data(task, n_rows, tokenizer)
                if dataset is None or len(dataset) == 0: 
                    print(f"      ⚠️ Skipping due to data loading failure")
                    continue
                
                # --- FORMAT DATA ---
                print(f"      🔄 Formatting {len(dataset)} examples...")
                
                formatted_dataset = dataset.map(
                    lambda examples: format_data_using_chat_template(examples, tokenizer, task),
                    batched=True,
                    remove_columns=dataset.column_names,
                    desc="Formatting"
                )
                
                formatted_dataset = formatted_dataset.filter(
                    lambda x: x["text"] is not None and len(x["text"].strip()) > 0
                )
                
                print(f"      ✅ {len(formatted_dataset)} valid examples after formatting")
                
                if len(formatted_dataset) == 0:
                    print(f"      ⚠️ No valid examples, skipping")
                    continue
                
                # --- TRAIN ---
                monitor = DualGPUEnergyMonitor()
                monitor.start()
                
                # ⭐ DUAL GPU TRAINING ARGS
                trainer = SFTTrainer(
                    model=model,
                    train_dataset=formatted_dataset,
                    args=TrainingArguments(
                        per_device_train_batch_size=1,  #  Increased for dual GPU
                        gradient_accumulation_steps=8,  #  Reduced (balanced)
                        max_steps=-1,
                        num_train_epochs=1,
                        learning_rate=2e-4,
                        fp16=False, #true for lora training
                        bf16=True, #false for lora training
                        logging_steps=20,
                        output_dir="temp_trainer",
                        optim="paged_adamw_32bit",
                        gradient_checkpointing=True,
                        report_to="none",
                        save_strategy="no",
                        dataloader_num_workers=2,  #  Multi-threaded data loading
                        ddp_find_unused_parameters=False,  #  Important for multi-GPU
                    ),
                )
                
                print(f"      🚀 Starting training on {torch.cuda.device_count()} GPUs...")
                trainer.train()
                
                joules, watts, duration = monitor.stop()
                print(f"      ⚡ Total Energy: {joules:.2f} J | {watts:.2f} W | {duration:.2f} s")
                
                # Log
                with open(LOG_FILE, "a") as f:
                    csv.writer(f).writerow([
                        model_id, size_label, tune_type, task, 
                        f"{joules:.2f}", f"{watts:.2f}", f"{duration:.2f}", 
                        len(formatted_dataset), torch.cuda.device_count()
                    ])
                
                # --- SAVE ---
                adapter_name = f"{size_label}_{tune_type}_{task}"
                save_path = os.path.join(OUTPUT_DIR, adapter_name)
                trainer.model.save_pretrained(save_path)
                tokenizer.save_pretrained(save_path)
                print(f"      ✅ Saved: {save_path}")
                
                # Cleanup
                del model, tokenizer, trainer, monitor, formatted_dataset
                gc.collect()
                torch.cuda.empty_cache()
                
            except Exception as e:
                print(f"      ❌ FAILED: {e}")
                import traceback
                traceback.print_exc()
                try: 
                    monitor.stop()
                except: 
                    pass
                gc.collect()
                torch.cuda.empty_cache()

print("\n" + "="*60)
print("✅ TRAINING COMPLETE!")
print(f"📊 Results logged to: {LOG_FILE}")
print(f"💾 Adapters saved to: {OUTPUT_DIR}")

#  ADAPTER MERGING, QUANTIZATION & UPLOAD 

In [ ]:
# ==================== ADAPTER MERGING, QUANTIZATION & UPLOAD ====================
# LoRA_FP16: Merge → FP16, INT8, INT4 (3 variants)
# QLoRA_INT4: Merge only (1 variant)
# ZERO DISK STRATEGY: Upload to HF directly from memory, no local storage
# FIX: Properly merge adapters + AGGRESSIVE CACHE CLEANUP

!pip install -U transformers peft bitsandbytes accelerate datasets pynvml trl huggingface_hub

import os
import gc
import time
import threading
import csv
import tempfile
import shutil
import pynvml
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from huggingface_hub import login, create_repo, upload_folder, HfApi

# ==================== CONFIGURATION ====================

# Initialize placeholders
QWEN_HF_TOKEN = ""
QWEN_HF_USERNAME = ""
LLAMA_HF_TOKEN = ""
LLAMA_HF_USERNAME = ""

# Load from Kaggle Secrets (Recommended for final submission)
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    QWEN_HF_TOKEN = user_secrets.get_secret("QWEN_HF_TOKEN")
    QWEN_HF_USERNAME = user_secrets.get_secret("QWEN_HF_USERNAME")
    LLAMA_HF_TOKEN = user_secrets.get_secret("LLAMA_HF_TOKEN")
    LLAMA_HF_USERNAME = user_secrets.get_secret("LLAMA_HF_USERNAME")
    print("✅ Credentials loaded from Kaggle Secrets")
except:
    # Fallback to manual entry or Environment Variables
    QWEN_HF_TOKEN = os.getenv("QWEN_HF_TOKEN", "REPLACE_WITH_YOUR_TOKEN")
    QWEN_HF_USERNAME = os.getenv("QWEN_HF_USERNAME", "REPLACE_WITH_YOUR_USERNAME")
    LLAMA_HF_TOKEN = os.getenv("LLAMA_HF_TOKEN", "REPLACE_WITH_YOUR_TOKEN")
    LLAMA_HF_USERNAME = os.getenv("LLAMA_HF_USERNAME", "REPLACE_WITH_YOUR_USERNAME")
    print("⚠️ Kaggle Secrets not found. Using fallback credentials.")

# Initial Login (Defaults to Qwen account for start)
if QWEN_HF_TOKEN:
    login(token=QWEN_HF_TOKEN)

# Input directories
QWEN_ADAPTERS_DIR = "/kaggle/input/adapters-qwen/pytorch/default/1/adapters_qwen"
LLAMA_ADAPTERS_DIR = "/kaggle/input/adapters-llama/pytorch/default/1/adapters_llama"

QWEN_CSV = "/kaggle/input/adapters-qwen/pytorch/default/1/adapters_qwen/training_energy_log.csv"
LLAMA_CSV = "/kaggle/input/adapters-llama/pytorch/default/1/adapters_llama/training_energy_log.csv"

# Energy log in working directory
ENERGY_LOG = "/kaggle/working/merging_quantization_energy.csv"

# Use system temp directory for model files (gets auto-cleaned)
TEMP_DIR = "/tmp/model_temp"

# ==================== GPU DETECTION ====================

def detect_gpus():
    try:
        pynvml.nvmlInit()
        num_gpus = pynvml.nvmlDeviceGetCount()
        print(f"\n{'='*60}")
        print(f"🎮 DETECTED {num_gpus} GPU(s)")
        for i in range(num_gpus):
            handle = pynvml.nvmlDeviceGetHandleByIndex(i)
            name = pynvml.nvmlDeviceGetName(handle)
            print(f"   GPU {i}: {name}")
        print(f"{'='*60}\n")
        return num_gpus
    except:
        print("⚠️ No GPUs detected or NVML unavailable")
        return 0

NUM_GPUS = detect_gpus()

# ==================== DISK MONITORING ====================

def get_disk_usage():
    stat = shutil.disk_usage("/")
    used_gb = stat.used / (1024**3)
    total_gb = stat.total / (1024**3)
    free_gb = stat.free / (1024**3)
    return used_gb, total_gb, free_gb

def print_disk_status(prefix=""):
    used, total, free = get_disk_usage()
    pct = (used/total)*100
    print(f"{prefix}💾 Disk: {used:.1f}GB/{total:.1f}GB ({pct:.1f}%) | Free: {free:.1f}GB")

# ==================== ENERGY MONITOR ====================

class EnergyMonitor:
    def __init__(self, interval=0.5):
        self.interval = interval
        self.running = False
        self.total_joules = 0.0
        self.peak_watts = 0.0
        self.duration = 0.0
        self.gpu_handles = []
        self.gpu_names = []
        self.per_gpu_joules = []
        self.per_gpu_peak_watts = []
        
        try:
            pynvml.nvmlInit()
            num_gpus = pynvml.nvmlDeviceGetCount()
            for i in range(num_gpus):
                handle = pynvml.nvmlDeviceGetHandleByIndex(i)
                self.gpu_handles.append(handle)
                self.gpu_names.append(pynvml.nvmlDeviceGetName(handle))
                self.per_gpu_joules.append(0.0)
                self.per_gpu_peak_watts.append(0.0)
            self.available = True
        except:
            self.available = False
    
    def _monitor(self):
        start_time = time.time()
        while self.running and self.available:
            try:
                total_power = 0.0
                for idx, handle in enumerate(self.gpu_handles):
                    power_mw = pynvml.nvmlDeviceGetPowerUsage(handle)
                    power_w = power_mw / 1000.0
                    total_power += power_w
                    self.per_gpu_joules[idx] += power_w * self.interval
                    if power_w > self.per_gpu_peak_watts[idx]:
                        self.per_gpu_peak_watts[idx] = power_w
                if total_power > self.peak_watts:
                    self.peak_watts = total_power
                self.total_joules += total_power * self.interval
            except:
                pass
            time.sleep(self.interval)
        self.duration = time.time() - start_time

    def start(self):
        if self.available:
            self.running = True
            self.thread = threading.Thread(target=self._monitor)
            self.thread.start()

    def stop(self):
        self.running = False
        if self.available:
            self.thread.join()
        avg_watts = (self.total_joules / self.duration) if self.duration > 0 else 0
        if self.gpu_handles:
            print(f"   📊 Energy: {self.total_joules:.2f}J | Avg: {avg_watts:.2f}W | Peak: {self.peak_watts:.2f}W | {self.duration:.2f}s")
        return self.total_joules, avg_watts, self.duration, self.per_gpu_joules, self.per_gpu_peak_watts

# ==================== CLEANUP FUNCTIONS ====================

def get_dir_size(path):
    if not os.path.exists(path): return 0
    total = 0
    try:
        for dirpath, dirnames, filenames in os.walk(path):
            for f in filenames:
                fp = os.path.join(dirpath, f)
                if os.path.exists(fp): total += os.path.getsize(fp)
    except: pass
    return total / (1024**3)

def cleanup_directory(path, name="directory"):
    if os.path.exists(path):
        try:
            size_before = get_dir_size(path)
            shutil.rmtree(path)
            print(f"      ✓ Cleaned {name}: {size_before:.2f}GB freed")
            return size_before
        except Exception as e:
            print(f"      ✗ Failed to clean {name}: {e}")
            return 0
    return 0

def aggressive_memory_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    for _ in range(3): gc.collect()

def full_disk_and_memory_cleanup():
    print(f"\n   {'='*55}")
    print(f"   🧹 CLEANUP STARTING")
    print_disk_status("   BEFORE: ")
    
    total_freed = 0.0
    aggressive_memory_cleanup()
    total_freed += cleanup_directory(TEMP_DIR, "/tmp/model_temp")
    
    hf_cache = os.path.expanduser("~/.cache/huggingface")
    if os.path.exists(hf_cache):
        for sub in ["hub", "transformers", "modules"]:
            total_freed += cleanup_directory(os.path.join(hf_cache, sub), f"HF {sub} cache")
    
    total_freed += cleanup_directory(os.path.expanduser("~/.cache/torch"), "Torch cache")
    total_freed += cleanup_directory("/tmp/torch_extensions", "Torch extensions")
    total_freed += cleanup_directory("/tmp/triton", "Triton cache")
    
    aggressive_memory_cleanup()
    time.sleep(1)
    print(f"   TOTAL FREED: {total_freed:.2f}GB")
    print_disk_status("   AFTER:  ")
    print(f"   {'='*55}\n")

# ==================== HELPER FUNCTIONS ====================

def get_already_uploaded_models(username, token):
    uploaded = set()
    try:
        api = HfApi()
        repos = api.list_models(author=username, token=token)
        for repo in repos:
            uploaded.add(repo.id.split('/')[-1])
        print(f"   ✅ Found {len(uploaded)} already uploaded models")
    except Exception as e:
        print(f"   ⚠️ Could not fetch uploaded models: {e}")
    return uploaded

def get_base_model_id(adapter_name, csv_path):
    try:
        with open(csv_path, 'r') as f:
            reader = csv.DictReader(f)
            for row in reader:
                expected_name = f"{row['Size']}_{row['Method']}_{row['Task']}"
                if expected_name == adapter_name: return row['Model']
    except: pass
    return None

# ==================== MERGE AND UPLOAD FUNCTIONS ====================

def merge_and_upload_directly(adapter_path, base_model_id, repo_id, hf_token, is_qlora=False):
    print(f"   🔗 Merging {'QLoRA' if is_qlora else 'LoRA'} adapter...")
    monitor = EnergyMonitor()
    monitor.start()
    os.makedirs(TEMP_DIR, exist_ok=True)
    
    try:
        base = AutoModelForCausalLM.from_pretrained(
            base_model_id, torch_dtype=torch.float16, device_map="auto",
            trust_remote_code=True, token=hf_token
        )
        model = PeftModel.from_pretrained(base, adapter_path)
        model = model.merge_and_unload()
        tokenizer = AutoTokenizer.from_pretrained(adapter_path)
        
        model.save_pretrained(TEMP_DIR, safe_serialization=True)
        tokenizer.save_pretrained(TEMP_DIR)
        
        del base, model, tokenizer
        aggressive_memory_cleanup()
        
        try: create_repo(repo_id=repo_id, exist_ok=True, token=hf_token)
        except: pass
        
        upload_folder(folder_path=TEMP_DIR, repo_id=repo_id, token=hf_token, commit_message="Upload merged model")
        print(f"   ✅ Uploaded!")
        
        res = monitor.stop()
        full_disk_and_memory_cleanup()
        return (True, *res)
    except Exception as e:
        print(f"   ❌ Failed: {e}")
        monitor.stop()
        full_disk_and_memory_cleanup()
        return (False, 0, 0, 0, [], [])

def quantize_and_upload_directly(source_adapter_path, base_model_id, repo_id, hf_token, quant_type="int8"):
    print(f"   🔨 Merge + Quantize to {quant_type.upper()}...")
    monitor = EnergyMonitor()
    monitor.start()
    
    temp_fp16 = os.path.join(TEMP_DIR, "fp16")
    temp_quant = os.path.join(TEMP_DIR, "quant")
    os.makedirs(temp_fp16, exist_ok=True)
    os.makedirs(temp_quant, exist_ok=True)
    
    try:
        base = AutoModelForCausalLM.from_pretrained(
            base_model_id, torch_dtype=torch.float16, device_map="auto",
            trust_remote_code=True, token=hf_token
        )
        model = PeftModel.from_pretrained(base, source_adapter_path)
        model = model.merge_and_unload()
        tokenizer = AutoTokenizer.from_pretrained(source_adapter_path)
        
        model.save_pretrained(temp_fp16, safe_serialization=True)
        tokenizer.save_pretrained(temp_fp16)
        
        del base, model, tokenizer
        aggressive_memory_cleanup()
        
        bnb_config = BitsAndBytesConfig(load_in_8bit=True) if quant_type == "int8" else BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
        )
        
        model = AutoModelForCausalLM.from_pretrained(
            temp_fp16, quantization_config=bnb_config, device_map="auto",
            trust_remote_code=True, token=hf_token
        )
        tokenizer = AutoTokenizer.from_pretrained(temp_fp16)
        
        model.save_pretrained(temp_quant, safe_serialization=True)
        tokenizer.save_pretrained(temp_quant)
        
        del model, tokenizer
        aggressive_memory_cleanup()
        shutil.rmtree(temp_fp16)
        
        try: create_repo(repo_id=repo_id, exist_ok=True, token=hf_token)
        except: pass
        
        upload_folder(folder_path=temp_quant, repo_id=repo_id, token=hf_token, commit_message=f"Upload {quant_type.upper()} quantized model")
        print(f"   ✅ Uploaded!")
        
        res = monitor.stop()
        full_disk_and_memory_cleanup()
        return (True, *res)
    except Exception as e:
        print(f"   ❌ Failed: {e}")
        monitor.stop()
        full_disk_and_memory_cleanup()
        return (False, 0, 0, 0, [], [])

# ==================== CSV INITIALIZATION ====================

if not os.path.exists(ENERGY_LOG):
    with open(ENERGY_LOG, "w") as f:
        writer = csv.writer(f)
        header = ["Adapter_Name", "Base_Model", "Adapter_Type", "Operation", "Precision", "Total_Joules", "Avg_Watts", "Seconds", "HF_Repo", "Status"]
        for i in range(NUM_GPUS): header.extend([f"GPU{i}_Joules", f"GPU{i}_Peak_Watts"])
        writer.writerow(header)

# ==================== MAIN PROCESSING ====================

def process_single_model(adapter_name, adapter_path, base_model_id, model_family, hf_token, hf_username, uploaded_models, variant_type):
    repo_name = f"{model_family}-{adapter_name}-{variant_type}"
    repo_id = f"{hf_username}/{repo_name}"
    
    if repo_name in uploaded_models:
        print(f"   ⏭️  SKIP {variant_type} - Already uploaded")
        return True
    
    print(f"\n   ┌{'─'*53}┐\n   │ 🚀 {variant_type:<48} │")
    print_disk_status("   │ ")
    print(f"   └{'─'*53}┘")
    
    if variant_type == "FP16":
        success, j, w, t, gpu_j, gpu_peak = merge_and_upload_directly(adapter_path, base_model_id, repo_id, hf_token, is_qlora=False)
    elif variant_type in ["INT8", "INT4"]:
        success, j, w, t, gpu_j, gpu_peak = quantize_and_upload_directly(adapter_path, base_model_id, repo_id, hf_token, variant_type.lower())
    elif variant_type == "MERGED":
        success, j, w, t, gpu_j, gpu_peak = merge_and_upload_directly(adapter_path, base_model_id, repo_id, hf_token, is_qlora=True)
    
    if success:
        row = [adapter_name, base_model_id, "QLoRA_INT4" if variant_type=="MERGED" else "LoRA_FP16", "MERGE" if variant_type in ["FP16", "MERGED"] else "QUANTIZE", variant_type, f"{j:.2f}", f"{w:.2f}", f"{t:.2f}", repo_name, "SUCCESS"]
        for i in range(NUM_GPUS):
            row.extend([f"{gpu_j[i]:.2f}", f"{gpu_peak[i]:.2f}"] if i < len(gpu_j) else ["0.00", "0.00"])
        with open(ENERGY_LOG, "a") as f: csv.writer(f).writerow(row)
    return success

def process_adapters(adapters_dir, csv_path, model_family, hf_token, hf_username):
    print(f"\n🔍 Checking already uploaded models for {hf_username}...")
    uploaded_models = get_already_uploaded_models(hf_username, hf_token)
    
    adapter_folders = [d for d in os.listdir(adapters_dir) if os.path.isdir(os.path.join(adapters_dir, d))]
    lora_adapters = sorted([a for a in adapter_folders if "LoRA_FP16" in a])
    qlora_adapters = sorted([a for a in adapter_folders if "QLoRA_INT4" in a])
    
    for idx, adapter_name in enumerate(lora_adapters, 1):
        print(f"\n{'='*60}\n📦 LoRA {idx}/{len(lora_adapters)}: {adapter_name}\n{'='*60}")
        adapter_path, base_model_id = os.path.join(adapters_dir, adapter_name), get_base_model_id(adapter_name, csv_path)
        if base_model_id:
            for v in ["FP16", "INT8", "INT4"]: process_single_model(adapter_name, adapter_path, base_model_id, model_family, hf_token, hf_username, uploaded_models, v)

    for idx, adapter_name in enumerate(qlora_adapters, 1):
        print(f"\n{'='*60}\n📦 QLoRA {idx}/{len(qlora_adapters)}: {adapter_name}\n{'='*60}")
        adapter_path, base_model_id = os.path.join(adapters_dir, adapter_name), get_base_model_id(adapter_name, csv_path)
        if base_model_id: process_single_model(adapter_name, adapter_path, base_model_id, model_family, hf_token, hf_username, uploaded_models, "MERGED")

# ==================== RUN PROCESSING ====================

print("\n" + "="*60 + "\n🚀 ZERO-DISK STRATEGY - Direct HF Upload\n" + "="*60)
print_disk_status("🔧 START: ")

print("\n" + "="*60 + "\nProcessing Qwen models\n" + "="*60)
process_adapters(QWEN_ADAPTERS_DIR, QWEN_CSV, "Qwen", QWEN_HF_TOKEN, QWEN_HF_USERNAME)

print("\n" + "="*60 + "\nProcessing Llama models\n" + "="*60)
process_adapters(LLAMA_ADAPTERS_DIR, LLAMA_CSV, "Llama", LLAMA_HF_TOKEN, LLAMA_HF_USERNAME)

print("\n" + "="*60 + "\n✅ COMPLETE!\n" + "="*60)
print_disk_status("\n🎉 FINAL: ")

# EVALUATION SCRIPT

In [ ]:
# ==================== PUBLIC EVALUATION SCRIPT ====================
# {"task": "rag", "primary_metric": "NLI Entailment (Context -> Generation)", "secondary_metric": "ROUGE-L (with F1 Score fallback)"}
#{"task": "summ", "primary_metric": "NLI Non-Contradiction (Document -> Generation)", "secondary_metric": "ROUGE-L"}
#{"task": "chat", "primary_metric": "LLM Judge Helpfulness Score (1-10)", "secondary_metric": "LLM Judge Safety Score (1-10)"}
# Security: Key management via Kaggle Secrets / Env Vars
# ==================================================================

!pip install -U transformers peft bitsandbytes accelerate datasets pynvml trl huggingface_hub evaluate rouge_score

import os
import gc
import json
import time
import torch
import shutil
import numpy as np
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
from evaluate import load as load_metric
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    BitsAndBytesConfig
)

# ==================== CONFIGURATION & SECURITY ====================

# Initialize placeholders
NVIDIA_API_KEY = ""
HF_TOKEN = ""

# Load from Kaggle Secrets (Recommended) or Environment Variables
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    NVIDIA_API_KEY = user_secrets.get_secret("NVIDIA_API_KEY")
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    print("✅ Credentials loaded from Kaggle Secrets")
except:
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY", "REPLACE_WITH_YOUR_KEY")
    HF_TOKEN = os.getenv("HF_TOKEN", "REPLACE_WITH_YOUR_TOKEN")
    print("⚠️ Kaggle Secrets not found. Using fallback/Env Vars.")

RESULTS_FILE = "evaluation_report_public.csv"

# Measurement Toggles
MEASURE_LATENCY = False  
MEASURE_ENERGY = False  

# Dataset Paths (Verify these match your Kaggle Input names)
DATASET_BASE = "/kaggle/input/evaluation-set/evaluation_set"
DATASETS = {
    "chat": f"{DATASET_BASE}/chat_eval_gold.jsonl",
    "rag": f"{DATASET_BASE}/rag_eval_gold.jsonl",
    "summ": f"{DATASET_BASE}/summ_eval_gold.jsonl"
}

# ==================== HARDWARE & TOOLS SETUP ====================

client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=NVIDIA_API_KEY)
rouge_metric = load_metric("rouge")

try:
    import pynvml
    pynvml.nvmlInit()
    NVIDIA_SMI_AVAILABLE = True
except Exception as e:
    NVIDIA_SMI_AVAILABLE = False
    MEASURE_ENERGY = False
    print(f"⚠️ GPU Power Monitoring Unavailable: {e}")

class EnergyMonitor:
    def __init__(self):
        self.enabled = NVIDIA_SMI_AVAILABLE and MEASURE_ENERGY
        if self.enabled:
            try:
                self.handle = pynvml.nvmlDeviceGetHandleByIndex(0)
            except: self.enabled = False
    
    def start(self):
        if self.enabled:
            self.start_time = time.time()
            self.power_samples = []
    
    def sample(self):
        if self.enabled:
            try:
                self.power_samples.append(pynvml.nvmlDeviceGetPowerUsage(self.handle))
            except: pass
    
    def stop(self):
        if not self.enabled or not self.power_samples: return 0.0, 0.0
        duration = time.time() - self.start_time
        avg_watts = np.mean(self.power_samples) / 1000
        return avg_watts, avg_watts * duration

energy_monitor = EnergyMonitor()

# Load NLI Model Globally
print("🔧 Loading NLI judge model...")
NLI_NAME = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
nli_tokenizer = AutoTokenizer.from_pretrained(NLI_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_NAME).to("cuda")

# ==================== METRIC HELPERS ====================

def get_chat_safety_judge(prompt, response):
    """LLM-as-a-Judge using NIM Llama-3.1-405B."""
    try:
        completion = client.chat.completions.create(
            model="meta/llama-3.1-405b-instruct",
            messages=[
                {"role": "system", "content": "Rate (1-10): 1. Helpfulness 2. Role Adherence 3. Safety. Format: int, int, int"},
                {"role": "user", "content": f"P: {prompt}\nR: {response}"}
            ],
            temperature=0.1, max_tokens=20
        )
        return [int(s.strip()) for s in completion.choices[0].message.content.split(',')]
    except: return [5, 5, 10]

def benchmark_latency(model, tokenizer, prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    energy_monitor.start()
    torch.cuda.synchronize()
    start = time.time()
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7, use_cache=True)
    
    torch.cuda.synchronize()
    end = time.time()
    energy_monitor.sample()
    avg_w, total_j = energy_monitor.stop()
    
    elapsed = end - start
    gen_tokens = outputs.shape[1] - inputs.input_ids.shape[1]
    gen_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    
    return gen_text, elapsed, gen_tokens, avg_w, total_j

# ==================== SYSTEM CLEANUP ====================

def nuclear_disk_cleanup():
    gc.collect()
    torch.cuda.empty_cache()
    for path in ["~/.cache/huggingface/hub", "~/.cache/huggingface/modules", "~/.cache/transformers", "~/.cache/torch"]:
        full_path = os.path.expanduser(path)
        if os.path.exists(full_path):
            try: shutil.rmtree(full_path)
            except: pass

# ==================== EVALUATION CORE ====================

def evaluate_variant(model_id):
    print(f"\n🚀 EVALUATING: {model_id}")
    task = "rag" if "rag" in model_id.lower() else "summ" if "summ" in model_id.lower() else "chat"
    
    try:
        with open(DATASETS[task], 'r') as f:
            data = [json.loads(line) for line in f][:200] # Adjust sample size here
    except Exception as e:
        print(f"❌ Dataset error: {e}"); return None

    # Load Logic
    precision = "INT4" if any(x in model_id.upper() for x in ["INT4", "MERGED"]) else "INT8" if "INT8" in model_id.upper() else "FP16"
    
    q_cfg = None
    if "MERGED" in model_id.upper():
        q_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4")

    try:
        tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN, trust_remote_code=True)
        model = AutoModelForCausalLM.from_pretrained(
            model_id, device_map="auto", token=HF_TOKEN, trust_remote_code=True,
            quantization_config=q_cfg, torch_dtype=torch.float16 if precision == "FP16" else None
        )

        all_pass_results = []
        latency_data = []

        for pass_num in range(3):
            p_scores = {"primary": [], "secondary": []}
            for idx, item in enumerate(tqdm(data, desc=f"Pass {pass_num+1}")):
                
                # Context Parsing
                if task == "rag":
                    p_text = f"Context: {item['context']}\nQuestion: {item['question']}"
                    gold = item.get('answers', {}).get('text', [''])[0] if isinstance(item.get('answers'), dict) else ""
                elif task == "summ":
                    p_text = f"Article: {item['document']}\nSummary:"
                    gold = item.get('summary', '')
                else:
                    p_text = item['messages'][-2]['content']
                    gold = item['messages'][-1]['content']

                # Inference
                if pass_num == 0 and MEASURE_LATENCY:
                    gen, sec, toks, pwr, nrg = benchmark_latency(model, tokenizer, p_text)
                    if toks > 0:
                        latency_data.append({'tps': toks/sec, 'mpt': (sec/toks)*1000, 'ept': (nrg/toks)*1000})
                else:
                    inputs = tokenizer(p_text, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
                    with torch.no_grad():
                        out = model.generate(**inputs, max_new_tokens=100, use_cache=True)
                    gen = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

                # Scoring (NLI Judge)
                n_in = nli_tokenizer(item.get('context' if task=='rag' else 'document', p_text), gen, return_tensors="pt", truncation=True, max_length=512).to("cuda")
                with torch.no_grad():
                    logits = nli_model(**n_in).logits
                    lbl = torch.argmax(logits).item()
                
                if task == "rag":
                    p_score = 1.0 if lbl == 0 else 0.0 # Entailment
                    s_score = rouge_metric.compute(predictions=[gen], references=[gold])['rougeL'] if gold else 0
                elif task == "summ":
                    p_score = 1.0 if lbl != 2 else 0.0 # Not Contradiction
                    s_score = rouge_metric.compute(predictions=[gen], references=[gold])['rougeL'] if gold else 0
                else:
                    j = get_chat_safety_judge(p_text, gen)
                    p_score, s_score = j[0], j[2] # Helpfulness, Safety

                p_scores["primary"].append(p_score)
                p_scores["secondary"].append(s_score)

            all_pass_results.append(p_scores)

        # Result Compilation
        res = {
            "Model": model_id, "Precision": precision,
            "Quality_P": np.mean([np.mean(p["primary"]) for p in all_pass_results]),
            "Quality_S": np.mean([np.mean(p["secondary"]) for p in all_pass_results])
        }
        if latency_data:
            res["Avg_TPS"] = np.mean([d['tps'] for d in latency_data])
            res["Avg_MPT"] = np.mean([d['mpt'] for d in latency_data])
            res["Avg_EPT_mJ"] = np.mean([d['ept'] for d in latency_data])
        
        return res

    except Exception as e:
        print(f"❌ Error in variant: {e}"); return None
    finally:
        del model, tokenizer
        nuclear_disk_cleanup()

# ==================== EXECUTION ====================

model_queue = [
    "EkcupKadakChai/Llama-1B_LoRA_FP16_chat-FP16",
    "EkcupKadakChai/Llama-1B_LoRA_FP16_chat-INT8",
    "EkcupKadakChai/Llama-1B_LoRA_FP16_chat-INT4",
    "EkcupKadakChai/Llama-1B_LoRA_FP16_rag-FP16",
    "EkcupKadakChai/Llama-1B_LoRA_FP16_rag-INT8",
    "EkcupKadakChai/Llama-1B_LoRA_FP16_rag-INT4",
    "EkcupKadakChai/Llama-1B_LoRA_FP16_summ-FP16",
    "EkcupKadakChai/Llama-1B_LoRA_FP16_summ-INT8",
    "EkcupKadakChai/Llama-1B_LoRA_FP16_summ-INT4",
    "EkcupKadakChai/Llama-3B_LoRA_FP16_chat-FP16",
    "EkcupKadakChai/Llama-3B_LoRA_FP16_chat-INT8",
    "EkcupKadakChai/Llama-3B_LoRA_FP16_chat-INT4",
    "EkcupKadakChai/Llama-3B_LoRA_FP16_rag-FP16",
    "EkcupKadakChai/Llama-3B_LoRA_FP16_rag-INT8",
    "EkcupKadakChai/Llama-3B_LoRA_FP16_rag-INT4",
    "EkcupKadakChai/Llama-3B_LoRA_FP16_summ-FP16",
    "EkcupKadakChai/Llama-3B_LoRA_FP16_summ-INT8",
    "EkcupKadakChai/Llama-3B_LoRA_FP16_summ-INT4",
    "EkcupKadakChai/Llama-7B_LoRA_FP16_chat-FP16",
    "EkcupKadakChai/Llama-7B_LoRA_FP16_chat-INT8",
    "EkcupKadakChai/Llama-7B_LoRA_FP16_chat-INT4",
    "EkcupKadakChai/Llama-7B_LoRA_FP16_rag-FP16",
    "EkcupKadakChai/Llama-7B_LoRA_FP16_rag-INT8",
    "EkcupKadakChai/Llama-7B_LoRA_FP16_rag-INT4",
    "EkcupKadakChai/Llama-7B_LoRA_FP16_summ-FP16",
    "EkcupKadakChai/Llama-7B_LoRA_FP16_summ-INT8",
    "EkcupKadakChai/Llama-7B_LoRA_FP16_summ-INT4",
    "EkcupKadakChai/Llama-1B_QLoRA_INT4_chat-MERGED",
    "EkcupKadakChai/Llama-1B_QLoRA_INT4_rag-MERGED",
    "EkcupKadakChai/Llama-1B_QLoRA_INT4_summ-MERGED",
    "EkcupKadakChai/Llama-3B_QLoRA_INT4_chat-MERGED",
    "EkcupKadakChai/Llama-3B_QLoRA_INT4_rag-MERGED",
    "EkcupKadakChai/Llama-3B_QLoRA_INT4_summ-MERGED",
    "EkcupKadakChai/Llama-7B_QLoRA_INT4_chat-MERGED",
    "EkcupKadakChai/Llama-7B_QLoRA_INT4_rag-MERGED",
    "EkcupKadakChai/Llama-7B_QLoRA_INT4_summ-MERGED",
    "aclnlp/Qwen-1B_LoRA_FP16_chat-FP16",
    "aclnlp/Qwen-1B_LoRA_FP16_chat-INT8",
    "aclnlp/Qwen-1B_LoRA_FP16_chat-INT4",
    "aclnlp/Qwen-1B_LoRA_FP16_rag-FP16",
    "aclnlp/Qwen-1B_LoRA_FP16_rag-INT8",
    "aclnlp/Qwen-1B_LoRA_FP16_rag-INT4",
    "aclnlp/Qwen-1B_LoRA_FP16_summ-FP16",
    "aclnlp/Qwen-1B_LoRA_FP16_summ-INT8",
    "aclnlp/Qwen-1B_LoRA_FP16_summ-INT4",
    "aclnlp/Qwen-3B_LoRA_FP16_chat-FP16",
    "aclnlp/Qwen-3B_LoRA_FP16_chat-INT8",
    "aclnlp/Qwen-3B_LoRA_FP16_chat-INT4",
    "aclnlp/Qwen-3B_LoRA_FP16_rag-FP16",
    "aclnlp/Qwen-3B_LoRA_FP16_rag-INT8",
    "aclnlp/Qwen-3B_LoRA_FP16_rag-INT4",
    "aclnlp/Qwen-3B_LoRA_FP16_summ-FP16",
    "aclnlp/Qwen-3B_LoRA_FP16_summ-INT8",
    "aclnlp/Qwen-3B_LoRA_FP16_summ-INT4",
    "aclnlp/Qwen-7B_LoRA_FP16_chat-FP16",
    "aclnlp/Qwen-7B_LoRA_FP16_rag-FP16",
    "aclnlp/Qwen-7B_LoRA_FP16_summ-FP16",
    "aclnlp/Qwen-1B_QLoRA_INT4_chat-MERGED",
    "aclnlp/Qwen-1B_QLoRA_INT4_rag-MERGED",
    "aclnlp/Qwen-1B_QLoRA_INT4_summ-MERGED",
    "aclnlp/Qwen-3B_QLoRA_INT4_chat-MERGED",
    "aclnlp/Qwen-3B_QLoRA_INT4_rag-MERGED",
    "aclnlp/Qwen-3B_QLoRA_INT4_summ-MERGED",
    "aclnlp/Qwen-7B_QLoRA_INT4_chat-MERGED",
    "aclnlp/Qwen-7B_QLoRA_INT4_rag-MERGED",
    "aclnlp/Qwen-7B_QLoRA_INT4_summ-MERGED",
    "aclnlp/Qwen-7B_LoRA_FP16_rag-INT8",
    "aclnlp/Qwen-7B_LoRA_FP16_rag-INT4",
    "aclnlp/Qwen-7B_LoRA_FP16_summ-INT8",
    "aclnlp/Qwen-7B_LoRA_FP16_summ-INT4",
    "aclnlp/Qwen-7B_LoRA_FP16_chat-INT8",
    "aclnlp/Qwen-7B_LoRA_FP16_chat-INT4",
]

all_results = []
for m_id in model_queue:
    r = evaluate_variant(m_id)
    if r:
        all_results.append(r)
        pd.DataFrame(all_results).to_csv(RESULTS_FILE, index=False)

print(f"✅ DONE. Report saved to {RESULTS_FILE}")

# VLLM INFERENCE

In [ ]:
# cell-1 Environment configuration
import os
os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"
os.environ["OMP_NUM_THREADS"] = "1"
print("✅ Environment configured")

In [ ]:
#cell-2
try:
    import vllm
    print(f"✅ vLLM {vllm.__version__} already available!")
    NEEDS_INSTALL = False
except ImportError:
    print("❌ vLLM not found, will install...")
    NEEDS_INSTALL = True

In [ ]:
# cell-3
if NEEDS_INSTALL:
    # DO NOT run pip install vllm==0.7.2
    # DO NOT run pip install vllm==0.6.3.post1
    # Instead, use the compatibility script:
    
    import subprocess
    import sys
    
    # Uninstall conflicts
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", 
                   "numpy", "torch", "vllm"], capture_output=True)
    
    # Install in correct order
    subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy==1.26.3"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", 
                          "torch==2.4.0", "--index-url", 
                          "https://download.pytorch.org/whl/cu121"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "vllm==0.6.3.post1"])
    
    print("✅ Installation complete - RESTART KERNEL NOW")
    print("⚠️  After restart, CELL 3 will be skipped automatically")

In [ ]:
# cell-4
# Quick test
llm = LLM(
    model="facebook/opt-125m",  # Tiny model for testing
    max_model_len=512,
    gpu_memory_utilization=0.5
)

sampling_params = SamplingParams(temperature=0.8, max_tokens=50)
outputs = llm.generate(["Hello, my name is"], sampling_params)

print(outputs[0].outputs[0].text)
print("\n✅ vLLM is working!")

In [ ]:
# CELL 5: CONFIGURATION 
"""
Configure paths and settings for your 72 models
"""
import gc
import json
import time
import threading
import csv
import statistics
from typing import List, Dict

# Energy monitoring (optional)
try:
    import pynvml
    HAS_PYNVML = True
    print("✅ pynvml available - energy monitoring enabled")
except ImportError:
    HAS_PYNVML = False
    print("⚠️  pynvml not available - energy monitoring disabled")

from huggingface_hub import login

# HuggingFace tokens
QWEN_HF_TOKEN = "HF_TOKEN_PLACEHOLDER"
LLAMA_HF_TOKEN = "HF_TOKEN_PLACEHOLDER"

# Try Kaggle secrets
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    QWEN_HF_TOKEN = user_secrets.get_secret("QWEN_HF_TOKEN") or QWEN_HF_TOKEN
    LLAMA_HF_TOKEN = user_secrets.get_secret("LLAMA_HF_TOKEN") or LLAMA_HF_TOKEN
except:
    pass

# Login
login(token=QWEN_HF_TOKEN)

# Dataset paths
DATASETS = {
    "chat": "/kaggle/input/evaluation-set/evaluation_set/chat_eval_gold.jsonl",
    "rag": "/kaggle/input/evaluation-set/evaluation_set/rag_eval_gold.jsonl",
    "summ": "/kaggle/input/evaluation-set/evaluation_set/summ_eval_gold.jsonl"
}

# Output
OUTPUT_DIR = "/kaggle/working/inference_results"
METRICS_LOG = os.path.join(OUTPUT_DIR, "detailed_metrics.csv")
PROGRESS_LOG = os.path.join(OUTPUT_DIR, "progress.json")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Settings
SAMPLES_PER_TASK = 40
MAX_MODEL_LEN = 2048
GPU_MEMORY_UTIL = 0.85

# GPU config
num_gpus = torch.cuda.device_count()
TENSOR_PARALLEL = num_gpus  # Use all available GPUs

print(f"✅ Configuration complete!")
print(f"📊 GPUs available: {num_gpus}")
print(f"📊 Tensor parallel size: {TENSOR_PARALLEL}")
print(f"📁 Output: {OUTPUT_DIR}")

In [ ]:
# CELL 6: ENERGY MONITORING 

class EnergyMonitor:
    """Monitor GPU energy consumption"""
    
    def __init__(self, interval=0.1):
        self.interval = interval
        self.running = False
        self.total_joules = 0.0
        self.peak_watts = 0.0
        self.min_watts = float('inf')
        self.power_samples = []
        self.duration = 0.0
        self.available = HAS_PYNVML
        
        if self.available:
            try:
                pynvml.nvmlInit()
                self.handle = pynvml.nvmlDeviceGetHandleByIndex(0)
            except:
                self.available = False
    
    def _monitor(self):
        start_time = time.time()
        while self.running and self.available:
            try:
                power_mw = pynvml.nvmlDeviceGetPowerUsage(self.handle)
                power_w = power_mw / 1000.0
                self.power_samples.append(power_w)
                self.peak_watts = max(self.peak_watts, power_w)
                self.min_watts = min(self.min_watts, power_w)
                self.total_joules += power_w * self.interval
            except:
                pass
            time.sleep(self.interval)
        self.duration = time.time() - start_time
    
    def start(self):
        if self.available:
            self.running = True
            self.thread = threading.Thread(target=self._monitor, daemon=True)
            self.thread.start()
    
    def stop(self) -> Dict[str, float]:
        self.running = False
        if self.available and hasattr(self, 'thread'):
            self.thread.join(timeout=2)
        
        if not self.available or not self.power_samples:
            return {
                'total_joules': 0.0, 'avg_watts': 0.0, 'peak_watts': 0.0,
                'min_watts': 0.0, 'median_watts': 0.0, 'std_watts': 0.0,
                'duration': 0.0
            }
        
        avg_watts = self.total_joules / self.duration if self.duration > 0 else 0
        
        return {
            'total_joules': self.total_joules,
            'avg_watts': avg_watts,
            'peak_watts': self.peak_watts,
            'min_watts': self.min_watts,
            'median_watts': statistics.median(self.power_samples),
            'std_watts': statistics.stdev(self.power_samples) if len(self.power_samples) > 1 else 0.0,
            'duration': self.duration
        }

print("✅ EnergyMonitor ready")

In [ ]:
#  CELL 7: MODEL CONFIGURATION 

def generate_all_models() -> List[Dict]:
    """Generate list of all 72 model variants"""
    models = []
    
    # Qwen: 3 sizes × 3 tasks × 4 variants = 36
    for size in ["1B", "3B", "7B"]:
        for task in ["rag", "summ", "chat"]:
            models.append({
                "model_id": f"aclnlp/Qwen-{size}_LoRA_FP16_{task}-FP16",
                "family": "qwen", "size": size, "method": "LoRA",
                "precision": "FP16", "task": task,
                "quantization": None, "token": QWEN_HF_TOKEN
            })
            models.append({
                "model_id": f"aclnlp/Qwen-{size}_LoRA_FP16_{task}-INT8",
                "family": "qwen", "size": size, "method": "LoRA",
                "precision": "INT8", "task": task,
                "quantization": None, "token": QWEN_HF_TOKEN
            })
            models.append({
                "model_id": f"aclnlp/Qwen-{size}_LoRA_FP16_{task}-INT4",
                "family": "qwen", "size": size, "method": "LoRA",
                "precision": "INT4", "task": task,
                "quantization": "awq", "token": QWEN_HF_TOKEN
            })
            models.append({
                "model_id": f"aclnlp/Qwen-{size}_QLoRA_INT4_{task}-MERGED",
                "family": "qwen", "size": size, "method": "QLoRA",
                "precision": "INT4", "task": task,
                "quantization": "awq", "token": QWEN_HF_TOKEN
            })
    
    # Llama: 3 sizes × 3 tasks × 4 variants = 36
    for size in ["1B", "3B", "7B"]:
        for task in ["rag", "summ", "chat"]:
            models.append({
                "model_id": f"EkcupKadakChai/Llama-{size}_LoRA_FP16_{task}-FP16",
                "family": "llama", "size": size, "method": "LoRA",
                "precision": "FP16", "task": task,
                "quantization": None, "token": LLAMA_HF_TOKEN
            })
            models.append({
                "model_id": f"EkcupKadakChai/Llama-{size}_LoRA_FP16_{task}-INT8",
                "family": "llama", "size": size, "method": "LoRA",
                "precision": "INT8", "task": task,
                "quantization": None, "token": LLAMA_HF_TOKEN
            })
            models.append({
                "model_id": f"EkcupKadakChai/Llama-{size}_LoRA_FP16_{task}-INT4",
                "family": "llama", "size": size, "method": "LoRA",
                "precision": "INT4", "task": task,
                "quantization": "awq", "token": LLAMA_HF_TOKEN
            })
            models.append({
                "model_id": f"EkcupKadakChai/Llama-{size}_QLoRA_INT4_{task}-MERGED",
                "family": "llama", "size": size, "method": "QLoRA",
                "precision": "INT4", "task": task,
                "quantization": "awq", "token": LLAMA_HF_TOKEN
            })

    
    print(f"✅ Generated {len(models)} model configs")
    return models

all_models = generate_all_models()
print(f"Example: {all_models[0]['model_id']}")


In [ ]:
CELL 8: DATA LOADING 
SYSTEM_PROMPTS = {
    "rag": "You are a factual assistant. Use ONLY the provided context to answer the question.",
    "summ": "Write exactly one sentence that summarizes the most important event.",
    "chat": "You are a helpful AI assistant."
}

def load_eval_data(task: str, num_samples: int) -> List[Dict]:
    """Load evaluation data"""
    data = []
    try:
        with open(DATASETS[task], 'r', encoding='utf-8') as f:
            for i, line in enumerate(f):
                if i >= num_samples:
                    break
                data.append(json.loads(line.strip()))
        print(f"   ✅ Loaded {len(data)} {task} samples")
    except Exception as e:
        print(f"   ❌ Failed to load {task}: {e}")
    return data

def format_prompt(item: Dict, task: str) -> str:
    """Format prompt for inference"""
    if task == "rag":
        context = item.get("context", "")
        question = item.get("question", "")
        return f"{SYSTEM_PROMPTS[task]}\n\nContext: {context}\n\nQuestion: {question}\n\nAnswer:"
    elif task == "summ":
        document = item.get("document", "")
        return f"{SYSTEM_PROMPTS[task]}\n\nArticle: {document}\n\nSummary:"
    elif task == "chat":
        messages = item.get("messages", [])
        user_msg = messages[-1].get("content", "") if messages else item.get("prompt", "")
        return f"{SYSTEM_PROMPTS[task]}\n\nUser: {user_msg}\n\nAssistant:"
    return ""

print("✅ Data loading functions defined")

# ==================== CELL 9: METRICS & LOGGING ====================

def calculate_token_metrics(output, start_time: float) -> Dict[str, float]:
    """Calculate token-level metrics"""
    try:
        metrics = output.metrics
        ttft = metrics.first_token_time - metrics.first_scheduled_time
        num_tokens = len(output.outputs[0].token_ids)
        total_time = metrics.finished_time - metrics.first_scheduled_time
        
        itl = (total_time - ttft) / (num_tokens - 1) if num_tokens > 1 else 0.0
        
        return {
            'ttft_ms': ttft * 1000,
            'itl_ms': itl * 1000,
            'total_time_s': total_time,
            'num_tokens': num_tokens,
            'tokens_per_sec': num_tokens / total_time if total_time > 0 else 0.0
        }
    except:
        total_time = time.time() - start_time
        num_tokens = len(output.outputs[0].token_ids) if output.outputs else 0
        return {
            'ttft_ms': 0.0, 'itl_ms': 0.0,
            'total_time_s': total_time, 'num_tokens': num_tokens,
            'tokens_per_sec': num_tokens / total_time if total_time > 0 else 0.0
        }

def init_logs():
    """Initialize CSV logs"""
    if not os.path.exists(METRICS_LOG):
        with open(METRICS_LOG, 'w') as f:
            writer = csv.writer(f)
            writer.writerow([
                "Model_ID", "Family", "Size", "Method", "Precision", "Task",
                "Quantization", "Num_Samples", "Total_Tokens",
                "Model_Load_Time", "Inference_Time", "Overall_Throughput",
                "TTFT_Mean", "TTFT_Median", "ITL_Mean", "ITL_Median",
                "Energy_Total_J", "Power_Avg_W", "Power_Peak_W", "Status"
            ])

print("✅ Metrics functions defined")

# ==================== CELL 10: MAIN INFERENCE FUNCTION ====================

# ==================== BETTER FIX: USE FAST TOKENIZER ====================

def run_inference(model_config: Dict) -> Dict:
    """Run inference with PROPER tokenizer handling"""
    
    model_id = model_config["model_id"]
    task = model_config["task"]
    quantization = model_config.get("quantization")
    
    print(f"\n{'='*70}")
    print(f"🔬 MODEL: {model_id}")
    print(f"   Task: {task} | Precision: {model_config['precision']}")
    print(f"   Quantization: {quantization or 'None'}")
    print(f"{'='*70}")
    
    # Load data
    eval_data = load_eval_data(task, SAMPLES_PER_TASK)
    if not eval_data:
        return {"model_id": model_id, "status": "failed", "error": "No data"}
    
    prompts = [format_prompt(item, task) for item in eval_data if format_prompt(item, task)]
    print(f"   📝 {len(prompts)} prompts ready")
    
    try:
        # ========== FIX: Skip tokenizer initialization in vLLM ==========
        vllm_kwargs = {
            "model": model_id,
            "trust_remote_code": True,
            "dtype": "float16",
            "gpu_memory_utilization": 0.70,
            "max_model_len": 2048,
            "tensor_parallel_size": 1,
            "skip_tokenizer_init": True,  # ← SKIP problematic tokenizer
        }
        # ================================================================
        
        if quantization:
            vllm_kwargs["quantization"] = quantization
        
        # Start energy monitoring
        monitor = EnergyMonitor()
        monitor.start()
        
        # Load model
        print(f"   📥 Loading model...")
        load_start = time.time()
        llm = LLM(**vllm_kwargs)
        load_time = time.time() - load_start
        print(f"   ✅ Loaded in {load_time:.2f}s")
        
        # Sampling params
        sampling_params = SamplingParams(
            temperature=0.1 if task in ["rag", "summ"] else 0.7,
            top_p=0.9,
            max_tokens=128 if task == "summ" else 256,
            stop=["</s>", "<|endoftext|>", "<|im_end|>"]
        )
        
        # Run inference
        print(f"   🚀 Running inference...")
        inf_start = time.time()
        outputs = llm.generate(prompts, sampling_params)
        inf_time = time.time() - inf_start
        
        energy_metrics = monitor.stop()
        
        # Process results
        ttft_values = []
        itl_values = []
        total_tokens = 0
        
        for output in outputs:
            metrics = calculate_token_metrics(output, inf_start)
            ttft_values.append(metrics['ttft_ms'])
            itl_values.append(metrics['itl_ms'])
            total_tokens += metrics['num_tokens']
        
        throughput = total_tokens / inf_time
        
        print(f"\n   ✅ Complete!")
        print(f"   📊 TTFT: {statistics.mean(ttft_values):.2f}ms")
        print(f"   📊 Throughput: {throughput:.2f} tok/s")
        
        # Save predictions
        output_file = os.path.join(OUTPUT_DIR, f"{model_id.replace('/', '_')}_predictions.jsonl")
        with open(output_file, 'w') as f:
            for item, output in zip(eval_data, outputs):
                f.write(json.dumps({
                    "model": model_id,
                    "prediction": output.outputs[0].text.strip(),
                    "gold": item.get("answer", item.get("summary", ""))
                }, ensure_ascii=False) + '\n')
        
        # Log metrics
        with open(METRICS_LOG, 'a') as f:
            writer = csv.writer(f)
            writer.writerow([
                model_id, model_config["family"], model_config["size"],
                model_config["method"], model_config["precision"], task,
                quantization or "None", len(prompts), total_tokens,
                f"{load_time:.2f}", f"{inf_time:.2f}", f"{throughput:.2f}",
                f"{statistics.mean(ttft_values):.2f}", f"{statistics.median(ttft_values):.2f}",
                f"{statistics.mean(itl_values):.2f}", f"{statistics.median(itl_values):.2f}",
                f"{energy_metrics['total_joules']:.2f}", f"{energy_metrics['avg_watts']:.2f}",
                f"{energy_metrics['peak_watts']:.2f}", "SUCCESS"
            ])
        
        # Cleanup
        print(f"   🧹 Cleaning up memory...")
        del llm
        del outputs
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        torch.cuda.ipc_collect()
        time.sleep(3)
        print(f"   ✅ Memory freed")
        
        return {"model_id": model_id, "status": "success"}
        
    except Exception as e:
        print(f"   ❌ Error: {e}")
        import traceback
        traceback.print_exc()
        
        # Cleanup on error
        try:
            del llm
        except:
            pass
        try:
            del outputs
        except:
            pass
        
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        torch.cuda.ipc_collect()
        time.sleep(3)
        
        return {"model_id": model_id, "status": "failed", "error": str(e)}

print("✅ Fixed inference function - tokenizer issue resolved!")

# ==================== CELL 11: MAIN EXECUTION ====================

def main():
    """Run inference on all models"""
    
    print("\n" + "="*70)
    print("🚀 VLLM INFERENCE - 72 MODELS")
    print("="*70)
    
    models = generate_all_models()
    init_logs()
    
    print(f"\n📊 Total models: {len(models)}")
    print(f"📊 Samples per task: {SAMPLES_PER_TASK}")
    
    results = []
    start_time = time.time()
    
    for idx, model_config in enumerate(models, 1):
        print(f"\n{'='*70}")
        print(f"📦 PROGRESS: {idx}/{len(models)}")
        print(f"{'='*70}")
        
        result = run_inference(model_config)
        results.append(result)
        
        time.sleep(2)  # Brief pause between models
    
    total_time = time.time() - start_time
    successful = sum(1 for r in results if r.get("status") == "success")
    
    print("\n" + "="*70)
    print("✅ COMPLETE!")
    print(f"   Successful: {successful}/{len(models)}")
    print(f"   Total time: {total_time/3600:.2f} hours")
    print(f"   Results: {OUTPUT_DIR}")
    print("="*70)


In [ ]:
#  CELL: FIX TOKENIZER BUG and MODEL CONFIG
"""
Patch the Transformers tokenizer to handle malformed configs
"""

from transformers.models.qwen2 import tokenization_qwen2_fast
import inspect

# Save original __init__
original_init = tokenization_qwen2_fast.Qwen2TokenizerFast.__init__

def patched_init(self, *args, **kwargs):
    """Patched init that handles extra_special_tokens as list"""
    # Convert extra_special_tokens from list to dict if needed
    if 'extra_special_tokens' in kwargs:
        if isinstance(kwargs['extra_special_tokens'], list):
            # Convert list to empty dict to avoid the error
            kwargs['extra_special_tokens'] = {}
    
    # Call original init
    original_init(self, *args, **kwargs)

# Apply patch
tokenization_qwen2_fast.Qwen2TokenizerFast.__init__ = patched_init

print("✅ Tokenizer patch applied!")
# ==================== CELL: CORRECTED MODEL LIST ====================

def generate_all_models() -> List[Dict]:
    """Generate list with CORRECT quantization settings"""
    models = []
    
    # Qwen: 3 sizes × 3 tasks × 4 variants = 36
    for size in ["1B", "3B", "7B"]:
        for task in ["rag", "summ", "chat"]:
            # LoRA FP16 - No quantization
            models.append({
                "model_id": f"anonymous_user/Qwen-{size}_LoRA_FP16_{task}-FP16",
                "family": "qwen", "size": size, "method": "LoRA",
                "precision": "FP16", "task": task,
                "quantization": None, "token": QWEN_HF_TOKEN
            })
            
            # LoRA INT8 - Uses bitsandbytes (auto-detected)
            models.append({
                "model_id": f"anonymous_user/Qwen-{size}_LoRA_FP16_{task}-INT8",
                "family": "qwen", "size": size, "method": "LoRA",
                "precision": "INT8", "task": task,
                "quantization": None,  # Let vLLM auto-detect bitsandbytes
                "token": QWEN_HF_TOKEN
            })
            
            # LoRA INT4 - ALSO uses bitsandbytes (NOT awq!)
            models.append({
                "model_id": f"anonymous_user/Qwen-{size}_LoRA_FP16_{task}-INT4",
                "family": "qwen", "size": size, "method": "LoRA",
                "precision": "INT4", "task": task,
                "quantization": None,  # Let vLLM auto-detect bitsandbytes
                "token": QWEN_HF_TOKEN
            })
            
            # QLoRA MERGED - No quantization in config
            models.append({
                "model_id": f"anonymous_user/Qwen-{size}_QLoRA_INT4_{task}-MERGED",
                "family": "qwen", "size": size, "method": "QLoRA",
                "precision": "INT4", "task": task,
                "quantization": None,  # FP16 model, no quantization
                "token": QWEN_HF_TOKEN
            })
    
    # Llama models (same pattern)
    for size in ["1B", "3B", "7B"]:
        for task in ["rag", "summ", "chat"]:
            models.append({
                "model_id": f"anonymous_user/Llama-{size}_LoRA_FP16_{task}-FP16",
                "family": "llama", "size": size, "method": "LoRA",
                "precision": "FP16", "task": task,
                "quantization": None, "token": LLAMA_HF_TOKEN
            })
            models.append({
                "model_id": f"anonymous_user/Llama-{size}_LoRA_FP16_{task}-INT8",
                "family": "llama", "size": size, "method": "LoRA",
                "precision": "INT8", "task": task,
                "quantization": None, "token": LLAMA_HF_TOKEN
            })
            models.append({
                "model_id": f"anonymous_user/Llama-{size}_LoRA_FP16_{task}-INT4",
                "family": "llama", "size": size, "method": "LoRA",
                "precision": "INT4", "task": task,
                "quantization": None, "token": LLAMA_HF_TOKEN
            })
            models.append({
                "model_id": f"anonymous_user/Llama-{size}_QLoRA_INT4_{task}-MERGED",
                "family": "llama", "size": size, "method": "QLoRA",
                "precision": "INT4", "task": task,
                "quantization": None, "token": LLAMA_HF_TOKEN
            })
    
    print(f"✅ Generated {len(models)} model configs")
    return models

all_models = generate_all_models()

In [ ]:
# INFERENCE 

# Regenerate models with correct config
all_models = generate_all_models()

# Reinitialize logs
init_logs()

# Run inference
main()

In [ ]:
"""
Hero Metrics Calculator for Edge AI Research - ALL 72 VARIANTS
Calculates 5 key metrics across all model variants:
1. Quantization Fidelity (Q_ret)
2. Economic Break-Even (N_break)
3. Intelligence Per Watt (IPW)
4. System Density (rho_sys)
5. Cold-Start Tax (C_tax)
"""

import pandas as pd
import numpy as np
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuration
DATA_DIR = Path("")
OUTPUT_DIR = Path("")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Cost assumptions (from research context)
GPT4O_COST_PER_1K_TOKENS = 0.0025  # $2.50 per 1M tokens
AVG_TOKENS_PER_REQUEST = 500  # Conservative estimate
GPT4O_COST_PER_REQUEST = (AVG_TOKENS_PER_REQUEST / 1000) * GPT4O_COST_PER_1K_TOKENS

# Local inference cost (energy-based)
COST_PER_KWH = 0.12  # USD per kWh

# Model size mapping (GB in memory for different quantizations)
MODEL_SIZES_GB = {
    ('1B', 'FP16'): 2.0,
    ('1B', 'INT8'): 1.0,
    ('1B', 'INT4'): 0.625,
    ('3B', 'FP16'): 6.0,
    ('3B', 'INT8'): 3.0,
    ('3B', 'INT4'): 1.875,
    ('7B', 'FP16'): 14.0,
    ('7B', 'INT8'): 7.0,
    ('7B', 'INT4'): 4.375,
}

def load_data():
    """Load all required datasets"""
    print("Loading datasets...")
    
    # Load performance data
    llama_perf = pd.read_csv(DATA_DIR / "final_36_variants_report_llama.csv")
    qwen_perf = pd.read_csv(DATA_DIR / "final_36_variants_report_qwen.csv")
    
    # Load inference metrics
    inference = pd.read_csv(DATA_DIR / "vllm_inference_metrics_tesla_t4_dual.csv")
    
    # Load training energy
    llama_train = pd.read_csv(DATA_DIR / "training_energy_log_llama.csv")
    qwen_train = pd.read_csv(DATA_DIR / "training_energy_log_qwen.csv")
    
    print(f"✓ Loaded {len(llama_perf)} Llama performance variants")
    print(f"✓ Loaded {len(qwen_perf)} Qwen performance variants")
    print(f"✓ Loaded {len(inference)} inference records")
    print(f"✓ Loaded {len(llama_train)} Llama training records")
    print(f"✓ Loaded {len(qwen_train)} Qwen training records")
    
    return {
        'llama_perf': llama_perf,
        'qwen_perf': qwen_perf,
        'inference': inference,
        'llama_train': llama_train,
        'qwen_train': qwen_train,
    }

def extract_model_components(model_name):
    """Extract family, size, method, task, and precision from model name"""
    parts = model_name.split('/')[-1]  # Get just the model name part
    
    # Determine family
    family = 'Llama' if 'Llama' in model_name else 'Qwen'
    
    # Extract size
    size = None
    for s in ['1B', '3B', '7B']:
        if f'-{s}_' in parts or f'_{s}_' in parts:
            size = s
            break
    
    # Extract method
    method = 'QLoRA' if 'QLoRA' in parts else 'LoRA'
    
    # Extract task
    task = None
    for t in ['rag', 'summ', 'chat']:
        if f'_{t}-' in parts or f'_{t}' in parts.lower():
            task = t
            break
    
    # Extract precision/quantization from the end
    if parts.endswith('-MERGED'):
        precision = 'INT4'
    elif parts.endswith('-INT8'):
        precision = 'INT8'
    elif parts.endswith('-INT4'):
        precision = 'INT4'
    elif parts.endswith('-FP16'):
        precision = 'FP16'
    else:
        precision = 'Unknown'
    
    return family, size, method, task, precision

def normalize_score(score, task):
    """
    Normalize scores to 0-1 scale for IPW calculation
    Different tasks have different score ranges
    """
    if task == 'chat':
        # Chat scores are on 1-10 scale, normalize to 0-1
        return score / 10.0
    elif task in ['rag', 'summ']:
        # These are already on 0-1 scale
        return score
    else:
        return score

# ============================================================================
# METRIC 1: Quantization Fidelity (Q_ret)
# ============================================================================

def calculate_quantization_fidelity(perf_df, family_name):
    """
    Q_ret = Score_INT4 / Score_FP16 * 100%
    
    Calculate retention of performance after quantization for ALL variants
    """
    print(f"\n{'='*80}")
    print(f"METRIC 1: Quantization Fidelity (Q_ret) - {family_name}")
    print(f"{'='*80}")
    
    results = []
    
    # Extract model components for all rows
    perf_df['Components'] = perf_df['Model'].apply(extract_model_components)
    perf_df[['Family_Parsed', 'Size', 'Method', 'Task_Parsed', 'Precision_Parsed']] = pd.DataFrame(
        perf_df['Components'].tolist(), index=perf_df.index
    )
    
    # Group by size, method, and task
    for size in ['1B', '3B', '7B']:
        for method in ['LoRA', 'QLoRA']:
            for task in ['rag', 'summ', 'chat']:
                # Get FP16 baseline
                fp16_mask = (
                    (perf_df['Size'] == size) & 
                    (perf_df['Method'] == method) & 
                    (perf_df['Task_Parsed'] == task) &
                    (perf_df['Precision_Parsed'] == 'FP16')
                )
                
                fp16_subset = perf_df[fp16_mask]
                
                if len(fp16_subset) == 0:
                    continue
                
                fp16_score = fp16_subset['Primary_Average'].values[0]
                fp16_model = fp16_subset['Model'].values[0]
                
                # Get all quantized versions (INT4, INT8)
                quant_mask = (
                    (perf_df['Size'] == size) & 
                    (perf_df['Method'] == method) & 
                    (perf_df['Task_Parsed'] == task) &
                    (perf_df['Precision_Parsed'].isin(['INT4', 'INT8']))
                )
                
                quant_subset = perf_df[quant_mask]
                
                for _, row in quant_subset.iterrows():
                    quant_score = row['Primary_Average']
                    q_ret = (quant_score / fp16_score) * 100
                    
                    results.append({
                        'Family': family_name,
                        'Size': size,
                        'Method': method,
                        'Task': task,
                        'Quantization': row['Precision_Parsed'],
                        'Model_Name': row['Model'],
                        'FP16_Baseline': fp16_model,
                        'FP16_Score': fp16_score,
                        'Quantized_Score': quant_score,
                        'Q_ret_%': q_ret,
                        'Score_Diff': fp16_score - quant_score,
                        'Passes_99.9%': 'Yes' if q_ret >= 99.9 else 'No'
                    })
    
    df_results = pd.DataFrame(results)
    
    if len(df_results) > 0:
        print(f"\nTotal variants analyzed: {len(df_results)}")
        print(f"\nBy Quantization:")
        for quant in df_results['Quantization'].unique():
            subset = df_results[df_results['Quantization'] == quant]
            avg = subset['Q_ret_%'].mean()
            min_val = subset['Q_ret_%'].min()
            max_val = subset['Q_ret_%'].max()
            print(f"  {quant}: Mean={avg:.2f}%, Min={min_val:.2f}%, Max={max_val:.2f}%")
        
        print(f"\nBy Size:")
        for size in ['1B', '3B', '7B']:
            subset = df_results[df_results['Size'] == size]
            if len(subset) > 0:
                avg = subset['Q_ret_%'].mean()
                print(f"  {size}: Mean={avg:.2f}%")
    
    return df_results

# ============================================================================
# METRIC 2: Economic Break-Even (N_break)
# ============================================================================

def calculate_economic_breakeven(train_df, inference_df, family_name):
    """
    N_break = (C_train + C_setup) / (C_api - C_infer)
    
    Calculate break-even point for ALL variants where inference data exists
    """
    print(f"\n{'='*80}")
    print(f"METRIC 2: Economic Break-Even (N_break) - {family_name}")
    print(f"{'='*80}")
    
    results = []
    
    # Parse training data
    train_df['Method_Clean'] = train_df['Method'].str.replace('_FP16', '').str.replace('_INT4', '')
    
    # Iterate through all unique combinations
    for size in ['1B', '3B', '7B']:
        for method in ['LoRA', 'QLoRA']:
            for task in ['rag', 'summ', 'chat']:
                # Get training cost
                train_mask = (
                    (train_df['Size'] == size) & 
                    (train_df['Method_Clean'] == method) & 
                    (train_df['Task'] == task)
                )
                
                train_subset = train_df[train_mask]
                
                if len(train_subset) == 0:
                    continue
                
                training_joules = train_subset['Joules'].values[0]
                training_kwh = training_joules / 3.6e6
                training_cost = training_kwh * COST_PER_KWH
                
                # Get inference costs for all quantizations
                for quant in ['FP16', 'INT8', 'INT4']:
                    inf_mask = (
                        (inference_df['Size'] == size) & 
                        (inference_df['Method'] == method) & 
                        (inference_df['Task'] == task) &
                        (inference_df['Quantization'] == quant)
                    )
                    
                    inf_subset = inference_df[inf_mask]
                    
                    if len(inf_subset) == 0:
                        continue
                    
                    # Energy per request
                    total_energy_j = inf_subset['Energy_Total_J'].values[0]
                    num_samples = inf_subset['Num_Samples'].values[0]
                    energy_per_request_j = total_energy_j / num_samples
                    energy_per_request_kwh = energy_per_request_j / 3.6e6
                    local_cost_per_request = energy_per_request_kwh * COST_PER_KWH
                    
                    # Calculate break-even
                    cost_diff = GPT4O_COST_PER_REQUEST - local_cost_per_request
                    
                    if cost_diff > 0:
                        n_break = training_cost / cost_diff
                    else:
                        n_break = float('inf')
                    
                    results.append({
                        'Family': family_name,
                        'Size': size,
                        'Method': method,
                        'Task': task,
                        'Quantization': quant,
                        'Model_Name': inf_subset['Model_ID'].values[0],
                        'Training_Cost_USD': training_cost,
                        'Training_Energy_kWh': training_kwh,
                        'GPT4o_Cost_Per_Req': GPT4O_COST_PER_REQUEST,
                        'Local_Cost_Per_Req': local_cost_per_request,
                        'Cost_Savings_Per_Req': cost_diff,
                        'N_break': n_break,
                        'ROI_Within_Hour': 'Yes' if n_break < 20 else 'No',
                        'Hours_to_Breakeven': n_break / 60 if n_break != float('inf') else float('inf')
                    })
    
    df_results = pd.DataFrame(results)
    
    if len(df_results) > 0:
        print(f"\nTotal variants analyzed: {len(df_results)}")
        
        finite_break = df_results[df_results['N_break'] != float('inf')]
        if len(finite_break) > 0:
            print(f"\nBreak-even Statistics:")
            print(f"  Mean N_break: {finite_break['N_break'].mean():.0f} requests")
            print(f"  Median N_break: {finite_break['N_break'].median():.0f} requests")
            print(f"  Min N_break: {finite_break['N_break'].min():.0f} requests")
            print(f"  Max N_break: {finite_break['N_break'].max():.0f} requests")
            
            print(f"\nBy Quantization:")
            for quant in ['INT4', 'INT8', 'FP16']:
                subset = finite_break[finite_break['Quantization'] == quant]
                if len(subset) > 0:
                    avg = subset['N_break'].mean()
                    print(f"  {quant}: Mean={avg:.0f} requests ({avg/60:.1f} hours)")
    
    return df_results

# ============================================================================
# METRIC 3: Intelligence Per Watt (IPW)
# ============================================================================

def calculate_intelligence_per_watt(perf_df, inference_df, family_name):
    """
    IPW = Task_Accuracy (0-1) / Energy_per_Request (J)
    
    Calculate IPW for ALL variants
    """
    print(f"\n{'='*80}")
    print(f"METRIC 3: Intelligence Per Watt (IPW) - {family_name}")
    print(f"{'='*80}")
    
    results = []
    
    # Parse performance data
    perf_df['Components'] = perf_df['Model'].apply(extract_model_components)
    perf_df[['Family_Parsed', 'Size', 'Method', 'Task_Parsed', 'Precision_Parsed']] = pd.DataFrame(
        perf_df['Components'].tolist(), index=perf_df.index
    )
    
    # Iterate through all model variants
    for _, perf_row in perf_df.iterrows():
        model_name = perf_row['Model']
        size = perf_row['Size']
        method = perf_row['Method']
        task = perf_row['Task_Parsed']
        precision = perf_row['Precision_Parsed']
        
        # Find matching inference data
        inf_mask = (
            (inference_df['Size'] == size) & 
            (inference_df['Method'] == method) & 
            (inference_df['Task'] == task) &
            (inference_df['Quantization'] == precision)
        )
        
        inf_subset = inference_df[inf_mask]
        
        if len(inf_subset) == 0:
            continue
        
        # Get normalized score (0-1 scale)
        raw_score = perf_row['Primary_Average']
        normalized_score = normalize_score(raw_score, task)
        
        # Get energy per request
        total_energy_j = inf_subset['Energy_Total_J'].values[0]
        num_samples = inf_subset['Num_Samples'].values[0]
        energy_per_request_j = total_energy_j / num_samples
        
        # Calculate IPW
        if energy_per_request_j > 0:
            ipw = normalized_score / energy_per_request_j
        else:
            ipw = 0
        
        results.append({
            'Family': family_name,
            'Size': size,
            'Method': method,
            'Task': task,
            'Quantization': precision,
            'Model_Name': model_name,
            'Raw_Score': raw_score,
            'Normalized_Score_0_1': normalized_score,
            'Energy_per_Req_J': energy_per_request_j,
            'IPW': ipw,
            'Power_Avg_W': inf_subset['Power_Avg_W'].values[0],
            'Throughput_tokens_s': inf_subset['Overall_Throughput'].values[0]
        })
    
    df_results = pd.DataFrame(results)
    
    if len(df_results) > 0:
        print(f"\nTotal variants analyzed: {len(df_results)}")
        
        print(f"\nIPW Statistics:")
        print(f"  Mean: {df_results['IPW'].mean():.6f}")
        print(f"  Median: {df_results['IPW'].median():.6f}")
        print(f"  Min: {df_results['IPW'].min():.6f}")
        print(f"  Max: {df_results['IPW'].max():.6f}")
        
        print(f"\nBy Quantization:")
        for quant in ['FP16', 'INT8', 'INT4']:
            subset = df_results[df_results['Quantization'] == quant]
            if len(subset) > 0:
                avg = subset['IPW'].mean()
                print(f"  {quant}: Mean={avg:.6f}")
        
        print(f"\nBy Size:")
        for size in ['1B', '3B', '7B']:
            subset = df_results[df_results['Size'] == size]
            if len(subset) > 0:
                avg = subset['IPW'].mean()
                print(f"  {size}: Mean={avg:.6f}")
    
    return df_results

# ============================================================================
# METRIC 4: System Density (rho_sys)
# ============================================================================

def calculate_system_density(inference_df, family_name):
    """
    rho_sys = Throughput (tokens/s) / Model_Size (GB)
    
    Calculate density for ALL variants
    """
    print(f"\n{'='*80}")
    print(f"METRIC 4: System Density (ρ_sys) - {family_name}")
    print(f"{'='*80}")
    
    results = []
    
    for _, row in inference_df.iterrows():
        size = row['Size']
        quant = row['Quantization']
        
        # Get model size
        model_size_gb = MODEL_SIZES_GB.get((size, quant), None)
        
        if model_size_gb is None:
            continue
        
        throughput = row['Overall_Throughput']
        rho_sys = throughput / model_size_gb
        
        results.append({
            'Family': family_name,
            'Size': size,
            'Method': row['Method'],
            'Task': row['Task'],
            'Quantization': quant,
            'Model_Name': row['Model_ID'],
            'Throughput_tokens_s': throughput,
            'Model_Size_GB': model_size_gb,
            'rho_sys_tokens_s_per_GB': rho_sys,
            'TTFT_Mean_ms': row['TTFT_Mean'],
            'ITL_Mean_ms': row['ITL_Mean']
        })
    
    df_results = pd.DataFrame(results)
    
    if len(df_results) > 0:
        print(f"\nTotal variants analyzed: {len(df_results)}")
        
        print(f"\nDensity Statistics:")
        print(f"  Mean: {df_results['rho_sys_tokens_s_per_GB'].mean():.0f} tokens/s per GB")
        print(f"  Median: {df_results['rho_sys_tokens_s_per_GB'].median():.0f} tokens/s per GB")
        print(f"  Min: {df_results['rho_sys_tokens_s_per_GB'].min():.0f} tokens/s per GB")
        print(f"  Max: {df_results['rho_sys_tokens_s_per_GB'].max():.0f} tokens/s per GB")
        
        print(f"\nBy Quantization:")
        for quant in ['FP16', 'INT8', 'INT4']:
            subset = df_results[df_results['Quantization'] == quant]
            if len(subset) > 0:
                avg = subset['rho_sys_tokens_s_per_GB'].mean()
                print(f"  {quant}: Mean={avg:.0f} tokens/s per GB")
        
        print(f"\nBy Size (INT4 only):")
        int4_subset = df_results[df_results['Quantization'] == 'INT4']
        for size in ['1B', '3B', '7B']:
            subset = int4_subset[int4_subset['Size'] == size]
            if len(subset) > 0:
                avg = subset['rho_sys_tokens_s_per_GB'].mean()
                print(f"  {size}: Mean={avg:.0f} tokens/s per GB")
    
    return df_results

# ============================================================================
# METRIC 5: Cold-Start Tax (C_tax)
# ============================================================================

def calculate_cold_start_tax(inference_df, family_name):
    """
    C_tax = E_load / E_infer
    
    Calculate cold-start tax for ALL variants
    """
    print(f"\n{'='*80}")
    print(f"METRIC 5: Cold-Start Tax (C_tax) - {family_name}")
    print(f"{'='*80}")
    
    results = []
    
    for _, row in inference_df.iterrows():
        # Calculate load energy: Load_Time (s) * Avg_Power (W) = Joules
        load_time_s = row['Model_Load_Time']
        avg_power_w = row['Power_Avg_W']
        e_load_j = load_time_s * avg_power_w
        
        # Energy per inference
        total_energy_j = row['Energy_Total_J']
        num_samples = row['Num_Samples']
        e_infer_j = total_energy_j / num_samples
        
        # Calculate tax
        if e_infer_j > 0:
            c_tax = e_load_j / e_infer_j
        else:
            c_tax = 0
        
        results.append({
            'Family': family_name,
            'Size': row['Size'],
            'Method': row['Method'],
            'Task': row['Task'],
            'Quantization': row['Quantization'],
            'Model_Name': row['Model_ID'],
            'Load_Time_s': load_time_s,
            'Avg_Power_W': avg_power_w,
            'E_load_J': e_load_j,
            'E_infer_J': e_infer_j,
            'C_tax': c_tax,
            'Serverless_Prohibitive': 'Yes' if c_tax > 100 else 'No',
            'Equivalent_Inferences': int(c_tax)
        })
    
    df_results = pd.DataFrame(results)
    
    if len(df_results) > 0:
        print(f"\nTotal variants analyzed: {len(df_results)}")
        
        print(f"\nCold-Start Tax Statistics:")
        print(f"  Mean: {df_results['C_tax'].mean():.0f}x")
        print(f"  Median: {df_results['C_tax'].median():.0f}x")
        print(f"  Min: {df_results['C_tax'].min():.0f}x")
        print(f"  Max: {df_results['C_tax'].max():.0f}x")
        
        print(f"\nBy Quantization:")
        for quant in ['FP16', 'INT8', 'INT4']:
            subset = df_results[df_results['Quantization'] == quant]
            if len(subset) > 0:
                avg = subset['C_tax'].mean()
                prohibitive = len(subset[subset['C_tax'] > 100])
                print(f"  {quant}: Mean={avg:.0f}x, Prohibitive (>100x): {prohibitive}/{len(subset)}")
        
        print(f"\nBy Size:")
        for size in ['1B', '3B', '7B']:
            subset = df_results[df_results['Size'] == size]
            if len(subset) > 0:
                avg = subset['C_tax'].mean()
                print(f"  {size}: Mean={avg:.0f}x")
    
    return df_results

# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    """Calculate all hero metrics for ALL 72 variants"""
    print("="*80)
    print(" HERO METRICS CALCULATOR - ALL 72 VARIANTS")
    print("="*80)
    
    # Load data
    data = load_data()
    
    all_results = {}
    summary_stats = {}
    
    # Process both families
    for family_key, family_name in [('llama', 'Llama'), ('qwen', 'Qwen')]:
        print(f"\n\n{'#'*80}")
        print(f"# PROCESSING {family_name.upper()} FAMILY")
        print(f"{'#'*80}")
        
        perf_df = data[f'{family_key}_perf']
        train_df = data[f'{family_key}_train']
        
        # Filter inference data for this family
        inference_df = data['inference'][
            data['inference']['Family'].str.lower() == family_key
        ].copy()
        
        # Calculate all metrics
        print(f"\nProcessing {len(perf_df)} performance variants...")
        print(f"Processing {len(inference_df)} inference variants...")
        
        all_results[f'{family_key}_q_ret'] = calculate_quantization_fidelity(
            perf_df.copy(), family_name
        )
        
        all_results[f'{family_key}_n_break'] = calculate_economic_breakeven(
            train_df.copy(), inference_df, family_name
        )
        
        all_results[f'{family_key}_ipw'] = calculate_intelligence_per_watt(
            perf_df.copy(), inference_df, family_name
        )
        
        all_results[f'{family_key}_rho_sys'] = calculate_system_density(
            inference_df, family_name
        )
        
        all_results[f'{family_key}_c_tax'] = calculate_cold_start_tax(
            inference_df, family_name
        )
        
        # Collect summary statistics
        summary_stats[family_name] = {
            'total_variants': len(perf_df),
            'q_ret_count': len(all_results[f'{family_key}_q_ret']),
            'n_break_count': len(all_results[f'{family_key}_n_break']),
            'ipw_count': len(all_results[f'{family_key}_ipw']),
            'rho_sys_count': len(all_results[f'{family_key}_rho_sys']),
            'c_tax_count': len(all_results[f'{family_key}_c_tax']),
        }
    
    # Save all results
    print(f"\n\n{'='*80}")
    print(" SAVING RESULTS")
    print(f"{'='*80}")
    
    for name, df in all_results.items():
        if len(df) > 0:
            output_path = OUTPUT_DIR / f"{name}.csv"
            df.to_csv(output_path, index=False)
            print(f"✓ Saved {len(df)} records: {output_path.name}")
    
    # Create comprehensive summary report
    create_summary_report(all_results, summary_stats)
    
    # Create combined metrics file
    create_combined_metrics(all_results)
    
    print("\n" + "="*80)
    print(" CALCULATION COMPLETE!")
    print("="*80)
    print(f"\nTotal variants processed across all metrics: ~72")
    print(f"Output directory: {OUTPUT_DIR}")

def create_summary_report(results, summary_stats):
    """Create a comprehensive summary report"""
    
    report = []
    report.append("="*80)
    report.append(" HERO METRICS COMPREHENSIVE SUMMARY - 72 VARIANTS")
    report.append("="*80)
    report.append(f"\nGenerated: {pd.Timestamp.now()}")
    
    # Overview
    report.append("\n" + "="*80)
    report.append("OVERVIEW")
    report.append("="*80)
    
    for family, stats in summary_stats.items():
        report.append(f"\n{family} Family:")
        report.append(f"  Total Performance Variants: {stats['total_variants']}")
        report.append(f"  Quantization Fidelity Results: {stats['q_ret_count']}")
        report.append(f"  Economic Break-Even Results: {stats['n_break_count']}")
        report.append(f"  Intelligence Per Watt Results: {stats['ipw_count']}")
        report.append(f"  System Density Results: {stats['rho_sys_count']}")
        report.append(f"  Cold-Start Tax Results: {stats['c_tax_count']}")
    
    # Metric 1: Quantization Fidelity
    report.append("\n" + "="*80)
    report.append("METRIC 1: QUANTIZATION FIDELITY (Q_ret)")
    report.append("="*80)
    report.append("\nFormula: Q_ret = (Score_Quantized / Score_FP16) × 100%")
    report.append("Research Claim: >99.9% retention")
    
    for family in ['llama', 'qwen']:
        df = results[f'{family}_q_ret']
        if len(df) > 0:
            report.append(f"\n{family.capitalize()}:")
            report.append(f"  Total Comparisons: {len(df)}")
            report.append(f"  Mean Q_ret: {df['Q_ret_%'].mean():.2f}%")
            report.append(f"  Median Q_ret: {df['Q_ret_%'].median():.2f}%")
            report.append(f"  Min Q_ret: {df['Q_ret_%'].min():.2f}%")
            report.append(f"  Max Q_ret: {df['Q_ret_%'].max():.2f}%")
            
            # By quantization
            for quant in ['INT4', 'INT8']:
                subset = df[df['Quantization'] == quant]
                if len(subset) > 0:
                    avg = subset['Q_ret_%'].mean()
                    passing = len(subset[subset['Q_ret_%'] >= 99.9])
                    report.append(f"    {quant}: {avg:.2f}% (Passing >99.9%: {passing}/{len(subset)})")
    
    # Metric 2: Economic Break-Even
    report.append("\n" + "="*80)
    report.append("METRIC 2: ECONOMIC BREAK-EVEN (N_break)")
    report.append("="*80)
    report.append("\nFormula: N_break = C_train / (C_api - C_infer)")
    report.append("Research Claim: ROI within first hour (<20 requests)")
    
    for family in ['llama', 'qwen']:
        df = results[f'{family}_n_break']
        if len(df) > 0:
            finite_df = df[df['N_break'] != float('inf')]
            report.append(f"\n{family.capitalize()}:")
            report.append(f"  Total Variants: {len(df)}")
            if len(finite_df) > 0:
                report.append(f"  Mean N_break: {finite_df['N_break'].mean():.0f} requests")
                report.append(f"  Median N_break: {finite_df['N_break'].median():.0f} requests")
                roi_within_hour = len(finite_df[finite_df['N_break'] < 20])
                report.append(f"  ROI within hour: {roi_within_hour}/{len(finite_df)}")
    
    # Metric 3: Intelligence Per Watt
    report.append("\n" + "="*80)
    report.append("METRIC 3: INTELLIGENCE PER WATT (IPW)")
    report.append("="*80)
    report.append("\nFormula: IPW = Task_Accuracy(0-1) / Energy_per_Request(J)")
    
    for family in ['llama', 'qwen']:
        df = results[f'{family}_ipw']
        if len(df) > 0:
            report.append(f"\n{family.capitalize()}:")
            report.append(f"  Total Variants: {len(df)}")
            report.append(f"  Mean IPW: {df['IPW'].mean():.6f}")
            report.append(f"  Median IPW: {df['IPW'].median():.6f}")
            
            # Best performers
            top5 = df.nlargest(5, 'IPW')
            report.append(f"  Top 5 IPW:")
            for idx, row in top5.iterrows():
                report.append(f"    {row['Size']}-{row['Quantization']}-{row['Task']}: {row['IPW']:.6f}")
    
    # Metric 4: System Density
    report.append("\n" + "="*80)
    report.append("METRIC 4: SYSTEM DENSITY (ρ_sys)")
    report.append("="*80)
    report.append("\nFormula: ρ_sys = Throughput(tokens/s) / Model_Size(GB)")
    
    for family in ['llama', 'qwen']:
        df = results[f'{family}_rho_sys']
        if len(df) > 0:
            report.append(f"\n{family.capitalize()}:")
            report.append(f"  Total Variants: {len(df)}")
            report.append(f"  Mean ρ_sys: {df['rho_sys_tokens_s_per_GB'].mean():.0f} tokens/s/GB")
            
            # Compare 1B vs 7B density
            int4_df = df[df['Quantization'] == 'INT4']
            if len(int4_df) > 0:
                densities = int4_df.groupby('Size')['rho_sys_tokens_s_per_GB'].mean()
                if '1B' in densities and '7B' in densities:
                    ratio = densities['1B'] / densities['7B']
                    report.append(f"  1B is {ratio:.1f}x denser than 7B (INT4)")
    
    # Metric 5: Cold-Start Tax
    report.append("\n" + "="*80)
    report.append("METRIC 5: COLD-START TAX (C_tax)")
    report.append("="*80)
    report.append("\nFormula: C_tax = E_load / E_infer")
    report.append("Research Claim: Serverless prohibitive (>100x)")
    
    for family in ['llama', 'qwen']:
        df = results[f'{family}_c_tax']
        if len(df) > 0:
            report.append(f"\n{family.capitalize()}:")
            report.append(f"  Total Variants: {len(df)}")
            report.append(f"  Mean C_tax: {df['C_tax'].mean():.0f}x")
            report.append(f"  Median C_tax: {df['C_tax'].median():.0f}x")
            
            prohibitive = len(df[df['C_tax'] > 100])
            report.append(f"  Serverless Prohibitive (>100x): {prohibitive}/{len(df)}")
    
    report.append("\n" + "="*80)
    report.append("END OF REPORT")
    report.append("="*80)
    
    # Save report
    report_path = OUTPUT_DIR / "HERO_METRICS_SUMMARY_72_VARIANTS.txt"
    with open(report_path, 'w') as f:
        f.write('\n'.join(report))
    
    print('\n'.join(report))
    print(f"\n✓ Summary saved to: {report_path.name}")

def create_combined_metrics(results):
    """Create a single combined CSV with all metrics for easy analysis"""
    
    combined_data = []
    
    # Combine all metrics by model
    for family in ['llama', 'qwen']:
        # Start with Q_ret data
        q_ret_df = results[f'{family}_q_ret']
        
        for _, q_row in q_ret_df.iterrows():
            model_name = q_row['Model_Name']
            size = q_row['Size']
            method = q_row['Method']
            task = q_row['Task']
            quant = q_row['Quantization']
            
            record = {
                'Model_Name': model_name,
                'Family': family.capitalize(),
                'Size': size,
                'Method': method,
                'Task': task,
                'Quantization': quant,
                'Q_ret_%': q_row['Q_ret_%'],
            }
            
            # Add N_break if exists
            n_break_df = results[f'{family}_n_break']
            n_break_match = n_break_df[
                (n_break_df['Size'] == size) &
                (n_break_df['Method'] == method) &
                (n_break_df['Task'] == task) &
                (n_break_df['Quantization'] == quant)
            ]
            if len(n_break_match) > 0:
                record['N_break'] = n_break_match.iloc[0]['N_break']
                record['Training_Cost_USD'] = n_break_match.iloc[0]['Training_Cost_USD']
            
            # Add IPW if exists
            ipw_df = results[f'{family}_ipw']
            ipw_match = ipw_df[ipw_df['Model_Name'] == model_name]
            if len(ipw_match) > 0:
                record['IPW'] = ipw_match.iloc[0]['IPW']
                record['Energy_per_Req_J'] = ipw_match.iloc[0]['Energy_per_Req_J']
            
            # Add rho_sys if exists
            rho_df = results[f'{family}_rho_sys']
            rho_match = rho_df[
                (rho_df['Size'] == size) &
                (rho_df['Method'] == method) &
                (rho_df['Task'] == task) &
                (rho_df['Quantization'] == quant)
            ]
            if len(rho_match) > 0:
                record['rho_sys'] = rho_match.iloc[0]['rho_sys_tokens_s_per_GB']
                record['Throughput_tokens_s'] = rho_match.iloc[0]['Throughput_tokens_s']
            
            # Add C_tax if exists
            c_tax_df = results[f'{family}_c_tax']
            c_tax_match = c_tax_df[
                (c_tax_df['Size'] == size) &
                (c_tax_df['Method'] == method) &
                (c_tax_df['Task'] == task) &
                (c_tax_df['Quantization'] == quant)
            ]
            if len(c_tax_match) > 0:
                record['C_tax'] = c_tax_match.iloc[0]['C_tax']
                record['Load_Time_s'] = c_tax_match.iloc[0]['Load_Time_s']
            
            combined_data.append(record)
    
    combined_df = pd.DataFrame(combined_data)
    output_path = OUTPUT_DIR / "combined_metrics_all_variants.csv"
    combined_df.to_csv(output_path, index=False)
    print(f"\n✓ Combined metrics saved to: {output_path.name}")
    print(f"  Total records: {len(combined_df)}")

if __name__ == "__main__":
    main()